# YOLOv8‑Poly‑Dist (TF) — The Whole Codebase, Explained Visually

A single, self‑contained tour of **`yolov8-poly-dist-tf`**: a TensorFlow 2 re‑implementation of YOLOv8 with **two
extensions** beyond plain object detection:

1. **Polygon (instance‑segmentation) heads** in the *PolyYOLO radial* format — 24 vertices, one per 15° bin.
2. **Per‑object distance estimation** (metres), trained from a separate dataset merged at the batch level.

Everything below runs the project's **actual code** (imported, not re‑implemented) on **real images**, so every shape
and number is what training really produces. Figures are saved as reusable PNGs under `notebooks/assets/explainer/`.

> **How to read this.** We follow the data: *picture → pipeline → model → predictions → targets → loss → optimiser →
> runtime*. Each part gives (a) the **intuition**, (b) a **diagram**, and (c) a **real‑code** cell. Source pointers are
> written as `file.py:line` so you can jump straight to them.

### The three configuration tiers (one codebase)
The same code trains three tiers, selected purely by YAML (`configs/experiments/yolo/`):

| Tier (YAML) | Active heads | Task |
|---|---|---|
| `yolov8_bbox.yaml` | `box` + `cls` | detection only |
| `yolov8_poly.yaml` | `box` + `cls` + polygon | detection + segmentation |
| **`yolov8_poly_dist.yaml`** | **all 6 heads** | detection + segmentation + distance |

This notebook uses the full **`yolov8_poly_dist`** tier so every component is visible.

## 1 · Setup — import the real code + the drawing toolkit

We import the actual modules and load the `yolov8_poly_dist` config. The helpers below only *adapt data formats*
(a coco128 loader matching `data_pipeline/tfds_decoders.py:PolygonDecoder`'s schema), *draw diagrams*, and *save
figures*. Anything that computes a target, prediction or loss is the **repo's own code**.

**Data.** We use **coco128‑seg** — 128 real COCO photos (internet images) with polygon + box GT — as a stand‑in for
the project's private datasets. The segmentation machinery is identical; only the *class taxonomy* differs (coco's 80
vs. the project's 39 placeholder classes in `configs/class_map.py`). We label with COCO names for readability.

In [ ]:
# === Imports + config (REAL repo code) ==================================================
import os, sys, math, glob, warnings
warnings.filterwarnings("ignore"); os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
CWD = os.getcwd(); REPO_ROOT = CWD
for _ in range(5):
    if os.path.isdir(os.path.join(REPO_ROOT, "data_pipeline")): break
    REPO_ROOT = os.path.dirname(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)

import numpy as np
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Polygon as MplPolygon, Rectangle
from PIL import Image, ImageDraw

from configs.yaml_loader import load_config
from configs.class_map import DETECTION_CLASSES
from models.yolo_v8 import build_yolov8
from data_pipeline.augmentations import resample_polygons
from data_pipeline.yolo_parser import V8ParserExtended

cfg = load_config(os.path.join(REPO_ROOT, "configs/experiments/yolo/yolov8_poly_dist.yaml"))
MC = cfg.task.model
ASSET_DIR = os.path.join(REPO_ROOT, "notebooks/assets/explainer")
os.makedirs(ASSET_DIR, exist_ok=True)
def _save(name):
    plt.gcf().savefig(os.path.join(ASSET_DIR, name + ".png"), dpi=130, bbox_inches="tight", facecolor="white")
print("TensorFlow", tf.__version__, "| assets →", os.path.relpath(ASSET_DIR, REPO_ROOT))
print("model: num_classes=%d input=%s poly_size=%d angle_step=%d" % (MC.num_classes, MC.input_size, MC.output_poly_size, MC.angle_step))
DATA_DIR = "/Users/sumanth/claude-work/datasets/coco128-seg"

TensorFlow 2.15.0 | assets → notebooks/assets/explainer
model: num_classes=39 input=[672, 672, 3] poly_size=24 angle_step=15


In [ ]:
# === Diagram toolkit (consistent visual language for all schematics) ====================
PAL = dict(input="#e8edf3", conv="#cfe0f5", block="#bcd4ef", pool="#ffe0b3", concat="#f6cfe0",
           out="#c9efd6", op="#eee6f7", note="#fffbe8", up="#d9f2ec", down="#fde2cf")
EC  = dict(input="#5b6b80", conv="#2f5d9e", block="#21487f", pool="#b6791f", concat="#9b2d5e",
           out="#2e8b57", op="#6a4ca0", note="#b6791f", up="#1f8a8a", down="#c0651f")

def node(ax, x, y, w, h, title, sub="", kind="block", fs=9, tcol="#11223a"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.05",
                 fc=PAL[kind], ec=EC[kind], lw=1.4))
    if sub:
        ax.text(x+w/2, y+h*0.62, title, ha="center", va="center", fontsize=fs, color=tcol, weight="bold")
        ax.text(x+w/2, y+h*0.27, sub, ha="center", va="center", fontsize=fs-1.5, color="#33475e")
    else:
        ax.text(x+w/2, y+h/2, title, ha="center", va="center", fontsize=fs, color=tcol)

def arrow(ax, x1, y1, x2, y2, label="", color="#3a4858", lw=1.7, style="-|>", ls="-", lcol=None):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle=style, mutation_scale=13,
                 color=color, lw=lw, ls=ls, shrinkA=2, shrinkB=2))
    if label: ax.text((x1+x2)/2, (y1+y2)/2, label, fontsize=7.5, color=lcol or color,
                       ha="center", va="center", bbox=dict(fc="white", ec="none", pad=0.4))

def slab(ax, x, y, w, h, title="", kind="block", d=2.0):
    # a 3D-ish feature-map block (front face + top/side) — represents an H×W×C tensor
    fc, ec = PAL[kind], EC[kind]
    ax.add_patch(MplPolygon([(x,y),(x+d,y+d),(x+w+d,y+d),(x+w,y)], closed=True, fc=fc, ec=ec, lw=1.1, alpha=0.65))
    ax.add_patch(MplPolygon([(x+w,y),(x+w+d,y+d),(x+w+d,y+h+d),(x+w,y+h)], closed=True, fc=fc, ec=ec, lw=1.1, alpha=0.85))
    ax.add_patch(Rectangle((x,y),w,h, fc=fc, ec=ec, lw=1.3))
    if title: ax.text(x+w/2, y+h/2, title, ha="center", va="center", fontsize=8, weight="bold", color="#11223a")
print("diagram toolkit ready")

diagram toolkit ready


In [ ]:
# === Helpers: coco loader (mirrors PolygonDecoder schema) + REAL radial encode/decode ===
COCO80 = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light',
 'fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow','elephant',
 'bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
 'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle',
 'wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich','orange','broccoli','carrot',
 'hot dog','pizza','donut','cake','chair','couch','potted plant','bed','dining table','toilet','tv','laptop',
 'mouse','remote','keyboard','cell phone','microwave','oven','toaster','sink','refrigerator','book','clock',
 'vase','scissors','teddy bear','hair drier','toothbrush']
def np_(t): return t.numpy() if hasattr(t, "numpy") else np.asarray(t)

def load_example(stem, data_dir=DATA_DIR, size=672):
    # coco128 -> the repo's decoded-example dict, image resized to size^2 (square, like training)
    img = np.asarray(Image.open(os.path.join(data_dir, "images/train2017", stem + ".jpg")).convert("RGB"))
    classes, polys = [], []
    for line in open(os.path.join(data_dir, "labels/train2017", stem + ".txt")):
        p = line.split()
        if len(p) < 7: continue
        cls = int(float(p[0])); xy = np.asarray(p[1:], np.float32); xy = xy[:(len(xy)//2)*2].reshape(-1, 2)
        if xy.shape[0] < 3: continue
        classes.append(cls); polys.append(xy)
    n = len(classes); boxes = np.zeros((n, 4), np.float32)
    for i, xy in enumerate(polys):
        x = np.clip(xy[:,0],0,1); y = np.clip(xy[:,1],0,1); boxes[i] = [y.min(), x.min(), y.max(), x.max()]
    maxc = max([xy.shape[0]*2 for xy in polys] or [2]); parr = np.full((n, maxc), -1.0, np.float32)
    for i, xy in enumerate(polys): parr[i, :xy.size] = xy.reshape(-1)
    img672 = tf.cast(tf.image.resize(tf.cast(img, tf.float32), [size, size], "bilinear"), tf.uint8).numpy()
    return {"image": img672, "boxes": boxes, "classes": np.asarray(classes, np.int64),
            "polys": parr, "stem": stem, "names": [COCO80[c] for c in classes]}

_PARSER = V8ParserExtended(output_size=[672,672], expanded_strides={'3':8,'4':16,'5':32}, levels=['3','4','5'])
def radial_encode(box_yxyx, poly_flat, resample=64, angle_step=15):
    # REAL: resample_polygons -> yolo_parser._preprocess_polygons_v2 -> [24,3] (dist, angle_offset, conf)
    b = tf.constant(box_yxyx[None, :], tf.float32)
    p = resample_polygons(tf.constant(poly_flat[None, :], tf.float32), resample)
    return _PARSER._preprocess_polygons_v2(b, p, angle_step).numpy().reshape(-1, 3)
def radial_decode(box_yxyx, radial, conf_thresh=0.4, angle_step=15):
    # exactly as detection_generator/polygon_metrics: theta=(i+offset)*angle_step
    cy = (box_yxyx[0]+box_yxyx[2])/2; cx = (box_yxyx[1]+box_yxyx[3])/2; pts = []
    for i in range(radial.shape[0]):
        dist, off, conf = radial[i]
        if conf < conf_thresh: continue
        th = math.radians((i + off) * angle_step); pts.append((cx + dist*math.cos(th), cy + dist*math.sin(th)))
    return np.array(pts, np.float32)

def draw_boxes(ax, boxes, names=None, color="#ff3b30", H=672, W=672, lw=1.8):
    for i, b in enumerate(boxes):
        y1,x1,y2,x2 = b
        ax.add_patch(mpatches.Rectangle((x1*W, y1*H), (x2-x1)*W, (y2-y1)*H, fill=False, ec=color, lw=lw))
        if names is not None:
            ax.text(x1*W, y1*H-3, names[i], color="white", fontsize=7, bbox=dict(fc=color, ec="none", pad=0.6))
def draw_polys(ax, polys, color="#34c759", H=672, W=672, lw=1.6, alpha=0.28):
    for row in polys:
        xy = row.reshape(-1, 2); xy = xy[xy[:,0] > -1.0]
        if xy.shape[0] < 3: continue
        ax.add_patch(MplPolygon(np.stack([xy[:,0]*W, xy[:,1]*H],1), closed=True, fill=True, fc=color, ec=color, lw=lw, alpha=alpha))
print("data helpers ready")

data helpers ready


In [ ]:
# === End-to-end block diagram ===========================================================
fig, ax = plt.subplots(figsize=(14, 6.8)); ax.set_xlim(0,100); ax.set_ylim(0,52); ax.axis("off")
ax.set_title("From a raw dataset image to updated model weights", fontsize=13, weight="bold")
# pipeline (top)
for i,(t,s) in enumerate([("TFDS","encoded JPEG"),("decode","+pre-resize 672²"),("Copy-Paste","p=0.2"),
        ("Mosaic","4→4 +perspective"),("flip","poly→radial"),("uint8 batch","det 128 + dist 16")]):
    node(ax, 1+i*16.2, 43, 14.5, 6.5, t, s, "input" if i==0 else "block", fs=8.5)
    if i: arrow(ax, 1+i*16.2-1.7, 46.2, 1+i*16.2, 46.2)
node(ax, 80, 35, 16, 5.2, "GPU colour aug", "HSV+Albu, /255", "op", fs=8.5); arrow(ax, 89, 43, 89, 40.2)
# model (middle)
node(ax, 8, 20, 17, 8.5, "BACKBONE", "CSPDarkNetV8-S", "block", fs=10)
node(ax, 30, 20, 17, 8.5, "DECODER", "FPN-PAN (C2f)", "block", fs=10)
node(ax, 52, 20, 17, 8.5, "HEAD", "6 branches", "block", fs=10)
arrow(ax, 84, 35, 33, 28.5, "image 672²", lcol="#2f5d9e"); arrow(ax, 25, 24, 30, 24); arrow(ax, 47, 24, 52, 24)
# split
node(ax, 74, 26, 22, 8, "LOSS (training)", "TAL + box/cls/dfl/dist/poly", "concat", fs=9)
node(ax, 74, 13, 22, 7, "detection_generator", "DFL decode + per-class NMS", "input", fs=8.5)
arrow(ax, 69, 25, 74, 30); arrow(ax, 69, 23, 74, 16.5)
# optimizer (bottom)
node(ax, 26, 3, 40, 7, "SGD+Nesterov  warmup→cosine  +  EMA shadow", "optimizers/", "out", fs=9.5)
arrow(ax, 82, 26, 56, 10, "∂loss/∂w", lcol="#9b2d5e"); arrow(ax, 46, 10, 16, 20, "update", lcol="#2e8b57")
plt.tight_layout()
_save('00_overview'); plt.show()

![00_overview](assets/explainer/00_overview.png)

<sub>The whole system on one page — read top (data) → middle (model) → bottom (learning).</sub>

## 🧭 Primer — the minimum ML you need (skip if you already know CNNs)

This notebook is written so a **complete beginner** can follow it. If terms like *convolution*, *loss* or *gradient*
are new, read this short primer once — everything afterwards reuses these ideas. Every picture here is **generated by
code** (no external images), so you can re‑run and tinker.

### What is this "model", really?
A neural network is just a **function with millions of tunable knobs** (called **weights** or **parameters**). You
feed in the pixels of an image; out come numbers that we *interpret* as boxes, class labels, polygons and distances.

```
        pixels  ──►  [ model: millions of weights ]  ──►  numbers we read as boxes / classes / polygons / distance
```

Nobody sets those millions of knobs by hand. Instead we **train**: show the model an image, let it guess, measure how
**wrong** the guess is (one number, the **loss**), and automatically nudge every knob a tiny bit in the direction that
makes the loss smaller. Repeat millions of times and the knobs settle into values that produce good guesses. Three
words appear everywhere below:

- **features** — intermediate summaries the network computes from the pixels (edges → textures → object parts → objects).
- **prediction** — the model's guess (a box, a class score, …).
- **loss** — a single number measuring how far the prediction is from the human‑labelled truth. **Training = make the loss small.**

### How a CNN looks at an image — convolution & feature maps
A **convolution** is a tiny **sliding window** (a *filter*, e.g. 3×3) that moves across the image; at each position it
multiplies the pixels under it by the filter's numbers and adds them up to produce **one output value**. One filter
detects one kind of pattern (a vertical edge, a patch of green…). A layer has **many filters**, so its output is a
**stack of feature maps** — one map per pattern. Stacking many such layers lets the network build up from edges to
whole objects. This is the **C**onvolutional **N**eural **N**etwork (**CNN**).

In [ ]:
# === Convolution sliding window + feature-map stacks ====================================
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
a = ax[0]; a.set_xlim(-0.5, 12); a.set_ylim(-0.5, 7.5); a.set_aspect('equal'); a.axis('off')
a.set_title("one convolution: a 3×3 filter slides, each stop → one output number", fontsize=10.5, weight='bold')
for r in range(7):
    for c in range(7):
        a.add_patch(Rectangle((c, r), 1, 1, fc="#eef2f7", ec="#aab6c4", lw=0.6))
for r in range(2,5):
    for c in range(2,5):
        a.add_patch(Rectangle((c, r), 1, 1, fc="#cfe0f5", ec="#2f5d9e", lw=1.4))
a.text(3.5, -0.9, "input pixels (one channel)", ha='center', fontsize=8.5, color="#555")
for r in range(5):
    for c in range(5):
        a.add_patch(Rectangle((c+8, r+1), 0.8, 0.8, fc="#e9f6ee", ec="#9fcfb4", lw=0.6))
a.add_patch(Rectangle((8+2.0, 1+2.0), 0.8, 0.8, fc="#34c759", ec="#0a7d2c", lw=1.4))
a.text(10, 0.2, "output feature map", ha='center', fontsize=8.5, color="#555")
arrow(a, 5.1, 3.5, 9.9, 3.4, "multiply+add\n(3×3 patch)", lcol="#9b2d5e")
b = ax[1]; b.set_xlim(0, 100); b.set_ylim(0, 40); b.axis('off')
b.set_title("many filters → a stack of feature maps; deeper layers = smaller but richer", fontsize=10.5, weight='bold')
slab(b, 4, 12, 16, 16, "image\n3 ch", "input", d=2.2)
slab(b, 34, 14, 12, 12, "layer 1\n32 maps", "conv", d=3.0)
slab(b, 60, 16, 9, 9, "layer 2\n128 maps", "block", d=3.6)
slab(b, 82, 17.5, 6, 6, "deep\n512 maps", "block", d=4.2)
for x1,x2 in [(22,34),(48,60),(71,82)]: arrow(b, x1, 20, x2, 20)
b.text(50, 4, "each arrow = several conv layers; resolution shrinks, number of feature maps (channels) grows", ha='center', fontsize=8, color="#555")
plt.tight_layout()
_save('primer_cnn'); plt.show()

![primer_cnn](assets/explainer/primer_cnn.png)

<sub>A CNN = stacks of sliding-window filters. Early maps catch edges; deep maps represent whole object parts.</sub>

### How the model learns — the training loop & gradient descent
Learning is a loop. The model guesses, we compute the **loss** (how wrong), and then **calculus** tells us, for every
single weight, which direction to nudge it to reduce the loss — these directions are the **gradients**. We take a
small step downhill (the step size is the **learning rate**) and repeat. This is **gradient descent**: imagine a ball
rolling down into the lowest point of a valley, where the loss is smallest.

In [ ]:
# === Training loop cycle + gradient-descent valley =====================================
fig, ax = plt.subplots(1, 2, figsize=(14, 4.3))
a = ax[0]; a.set_xlim(0,100); a.set_ylim(0,40); a.axis('off'); a.set_title("the training loop (repeated millions of times)", fontsize=10.5, weight='bold')
node(a, 2, 26, 20, 8, "image", "(a training example)", "input", fs=9)
node(a, 30, 26, 22, 8, "MODEL", "weights → prediction", "block", fs=9)
node(a, 60, 26, 22, 8, "compare to truth", "→ LOSS (how wrong)", "concat", fs=9)
node(a, 60, 8, 22, 8, "GRADIENTS", "which way to nudge\neach weight", "op", fs=8.5)
node(a, 30, 8, 22, 8, "UPDATE weights", "step downhill (lr)", "out", fs=8.5)
arrow(a, 22, 30, 30, 30); arrow(a, 52, 30, 60, 30); arrow(a, 71, 26, 71, 16, "calculus"); arrow(a, 60, 12, 52, 12); arrow(a, 41, 16, 41, 26, "repeat", lcol="#2e8b57")
b = ax[1]; xs=np.linspace(-3,3,100); b.plot(xs, xs**2, color="#9b2d5e", lw=2)
for px in [-2.4,-1.3,-0.5]: b.plot(px, px**2, 'o', color="#0a84ff", ms=9); b.annotate("", (px+0.5, (px+0.5)**2),(px,px**2), arrowprops=dict(arrowstyle="->", color="#0a84ff"))
b.plot(0,0,'*', color="#2e8b57", ms=18); b.text(0,0.6,"minimum\n(lowest loss)", ha='center', fontsize=9, color="#2e8b57")
b.set_title("gradient descent: roll the 'ball' downhill\nto where the loss is smallest", fontsize=10.5, weight='bold')
b.set_xlabel("a weight's value"); b.set_ylabel("loss"); b.set_yticks([])
plt.tight_layout()
_save('primer_train'); plt.show()

![primer_train](assets/explainer/primer_train.png)

<sub>Guess → measure wrongness (loss) → compute gradients → step downhill. The learning rate is the step size.</sub>

### Two more building blocks you'll see everywhere
**Activation functions** are simple curves applied to a neuron's output. They (a) let the network represent
*non‑straight* relationships, and (b) reshape a raw number into the *range we need*:
- **ReLU** `max(0,x)` — keep positives, zero out negatives (the network's main non‑linearity).
- **sigmoid** `1/(1+e⁻ˣ)` — squashes any number into **(0,1)**: perfect for a **probability / confidence**.
- **softplus** `log(1+eˣ)` — a smooth curve that is **always positive**: used for quantities that can't be negative (a radius/distance).
- **softmax** — turns a list of numbers into **probabilities that sum to 1**: "pick among options" (used by the box's DFL bins).

**Loss building blocks** — three ways to measure "wrong":
- **L1** `|pred−true|` — the plain distance you're off; calm about outliers.
- **L2** `(pred−true)²` — squares the error, so **big mistakes hurt much more**.
- **BCE** (binary cross‑entropy) — the **confidence** loss for yes/no questions: being **confidently wrong** is punished
  enormously, being unsure is punished mildly. This is what trains the class and confidence heads.

In [ ]:
# === Activation functions ==============================================================
xs = np.linspace(-5, 5, 200)
fig, ax = plt.subplots(1, 4, figsize=(16, 2.8))
ax[0].plot(xs, np.maximum(0, xs), color="#2e8b57", lw=2); ax[0].set_title("ReLU = max(0,x)\nkeep positives", fontsize=9.5)
ax[1].plot(xs, 1/(1+np.exp(-xs)), color="#0a84ff", lw=2); ax[1].set_title("sigmoid → (0,1)\nprobability / confidence", fontsize=9.5)
ax[2].plot(xs, np.log1p(np.exp(xs)), color="#b6791f", lw=2); ax[2].set_title("softplus → (0,∞)\nalways positive (radius)", fontsize=9.5)
v = np.array([2.0, 1.0, 0.2, -1.0]); sm = np.exp(v)/np.exp(v).sum()
ax[3].bar(range(4), sm, color="#9b2d5e"); ax[3].set_title("softmax → sums to 1\npick among options", fontsize=9.5); ax[3].set_xticks(range(4))
for a in ax[:3]: a.axhline(0, color="#ccc", lw=0.8); a.axvline(0, color="#ccc", lw=0.8)
plt.tight_layout()
_save('primer_activations'); plt.show()

![primer_activations](assets/explainer/primer_activations.png)

<sub>Activations reshape raw numbers into the range we need: confidence (sigmoid), positive radius (softplus), a choice (softmax).</sub>

In [ ]:
# === Loss building blocks: L1, L2, BCE =================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 3.0))
e = np.linspace(-3, 3, 200)
ax[0].plot(e, np.abs(e), color="#1f8a8a", lw=2); ax[0].set_title("L1 = |error|\nrobust, steady", fontsize=10); ax[0].set_xlabel("prediction − truth")
ax[1].plot(e, e**2, color="#9b2d5e", lw=2); ax[1].set_title("L2 = error²\nbig mistakes hurt more", fontsize=10); ax[1].set_xlabel("prediction − truth")
p = np.linspace(0.001, 0.999, 200)
ax[2].plot(p, -np.log(p), color="#0a84ff", lw=2); ax[2].set_title("BCE when truth = 1\nconfidently wrong = huge loss", fontsize=10); ax[2].set_xlabel("predicted probability"); ax[2].set_ylabel("loss")
plt.tight_layout()
_save('primer_losses'); plt.show()

![primer_losses](assets/explainer/primer_losses.png)

<sub>L1 vs L2 differ in how hard they punish big errors; BCE punishes confident mistakes on yes/no questions.</sub>

### IoU — how we measure "do two boxes match?"
**IoU** (Intersection‑over‑Union) is the **overlap area divided by the combined area** of two boxes. It is **0** when
they don't touch and **1** when they're identical. It is the core measure of localization quality — used in the loss
(CIoU), in target assignment, and in NMS.

In [ ]:
# === IoU illustration ==================================================================
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
cases = [([0.1,0.1,0.6,0.6],[0.5,0.5,0.95,0.95],"small overlap → low IoU"),
         ([0.15,0.15,0.7,0.7],[0.3,0.3,0.85,0.85],"good overlap → high IoU"),
         ([0.2,0.2,0.8,0.8],[0.22,0.22,0.82,0.82],"near-identical → IoU ≈ 1")]
for a,(A,B,lab) in zip(ax,cases):
    ix1,iy1,ix2,iy2 = max(A[0],B[0]),max(A[1],B[1]),min(A[2],B[2]),min(A[3],B[3])
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); ua=(A[2]-A[0])*(A[3]-A[1]); ub=(B[2]-B[0])*(B[3]-B[1]); iou=inter/(ua+ub-inter)
    if inter>0: a.add_patch(Rectangle((ix1,iy1),ix2-ix1,iy2-iy1, fc="#ffd23f", ec="none", alpha=0.7))
    a.add_patch(Rectangle((A[0],A[1]),A[2]-A[0],A[3]-A[1], fill=False, ec="#ff3b30", lw=2))
    a.add_patch(Rectangle((B[0],B[1]),B[2]-B[0],B[3]-B[1], fill=False, ec="#2e8b57", lw=2))
    a.set_xlim(0,1); a.set_ylim(0,1); a.set_aspect('equal'); a.invert_yaxis(); a.set_xticks([]); a.set_yticks([])
    a.set_title(f"{lab}\nIoU = {iou:.2f}", fontsize=10)
plt.suptitle("IoU = overlap (yellow) ÷ union  ·  0 = no match, 1 = perfect", fontsize=11, weight='bold')
plt.tight_layout()
_save('primer_iou'); plt.show()

![primer_iou](assets/explainer/primer_iou.png)

<sub>IoU turns 'how well do two boxes match?' into one number in [0,1] — used by the loss, the assigner, and NMS.</sub>

That's the whole toolkit. From here on, whenever you see *feature map, loss, gradient, sigmoid/softmax/softplus, BCE,
L1/L2,* or *IoU*, it means exactly what this primer showed. Now to the real system. 👇

## 2 · Ten real images — boxes + polygons, covering the variety

We pin **10 coco128‑seg images** chosen to span the cases the polygon machinery must survive. Each shows the **box GT**
(red, `yxyx`‑normalised) and **polygon GT** (green), after resizing to the training resolution **672²**.

| # | content | polygon edge case |
|---|---------|-------------------|
| 1 | tv | simplest — ~6‑vertex **rectangle** |
| 2 | laptop, cell phone | **rectangular** electronics |
| 3 | cat (annotation) | one clean contour |
| 4 | person, horse, plant | person + animal + plant |
| 5 | person, bicycle, dog | **thin structure** + pet |
| 6 | person, motorcycle | **high‑vertex** complex (≈179 pts) |
| 7 | couch, plant, remote, book | **furniture** + tiny object |
| 8 | 40‑object dining scene | **dense crowd** |
| 9 | sink, toothbrush | **very thin** object |
| 10 | elephants | **large organic** shape |

In [ ]:
# === Load and render the 10 pinned samples with REAL box + polygon GT ===================
SAMPLE_STEMS = ["000000000595","000000000387","000000000575","000000000049","000000000074",
                "000000000529","000000000446","000000000164","000000000208","000000000312"]
SAMPLES = [load_example(s) for s in SAMPLE_STEMS]
fig, axes = plt.subplots(2, 5, figsize=(16.5, 7.2))
for ax, ex in zip(axes.ravel(), SAMPLES):
    ax.imshow(ex["image"]); ax.set_xticks([]); ax.set_yticks([])
    draw_polys(ax, ex["polys"]); draw_boxes(ax, ex["boxes"], ex["names"])
    ax.set_title(f"…{ex['stem'][-3:]}  ·  {len(ex['classes'])} obj", fontsize=9)
fig.suptitle("10 real samples · red = box GT (yxyx norm) · green = polygon GT · resized to 672²", fontsize=12, weight="bold")
plt.tight_layout()
_save('02_samples'); plt.show()

![02_samples](assets/explainer/02_samples.png)

<sub>Real COCO photos with real box+polygon ground truth — the variety the model must handle.</sub>

## 3 · The polygon representation — PolyYOLO *radial* encoding

A segmentation polygon is a **variable‑length** list of `(x,y)` vertices. A CNN head must output a **fixed‑size,
per‑pixel** tensor. PolyYOLO bridges the gap by re‑expressing the contour in **polar coordinates around the box
centre** and quantising the angle into a fixed number of bins.

> **The idea in one sentence:** stand at the box centre, sweep a ray through **24 directions** (one every **15°**,
> because `angle_step=15` and `360/15=24`), and for each direction record **how far** the boundary is (radius),
> **exactly which angle** within that 15° slice the boundary vertex sits at (a sub‑bin offset), and **whether** the
> direction has a vertex at all (confidence).

That turns any polygon into a fixed **`[24, 3]`** tensor — `(dist, angle, conf)` per bin — which the head emits as
3 × 24 channels. The polygon takes **three forms** across the codebase:

In [ ]:
# === The three-format journey diagram ===================================================
fig, ax = plt.subplots(figsize=(13.5, 3.6)); ax.set_xlim(0,100); ax.set_ylim(0,26); ax.axis("off")
node(ax, 1, 9, 26, 9, "1) TFDS input", "[N, max_v+2] flat (x,y)\nnormalised, -1 padded", "input", fs=9)
node(ax, 37, 9, 26, 9, "2) PolyYOLO target", "[N, 72] = (dist,angle,conf)×24\nyolo_parser._preprocess_polygons_v2", "block", fs=9)
node(ax, 73, 9, 26, 9, "3) Cartesian", "[M, 2] pixel (x,y), conf≥0.4\ndetection_generator / eval / viz", "out", fs=9)
arrow(ax, 27, 13.5, 37, 13.5, "radial encode"); arrow(ax, 63, 13.5, 73, 13.5, "decode")
ax.text(50, 3, "training & loss use form (2);  visualisation & evaluation decode to form (3)", fontsize=9, ha="center", color="#555")
plt.tight_layout()
_save('03_three_formats'); plt.show()

![03_three_formats](assets/explainer/03_three_formats.png)

<sub>A polygon lives in three representations; the head learns to predict form (2).</sub>

### 3.1 A worked example, by hand and by the real encoder

Let's encode a **square** (centre `(0.5, 0.5)`, half‑width `0.15`) so the numbers are checkable. For a vertex at
position `(x, y)`, the encoder computes, relative to the box centre `(cx, cy)`:

```
dx = x - cx ,  dy = y - cy
radius      = sqrt(dx² + dy²)
vertex_angle = atan2(dy, dx)  in degrees, wrapped to [0, 360)
bin index i  = floor(vertex_angle / 15)
sub-bin off  = (vertex_angle - i·15) / 15      ∈ [0, 1)     # NOT a one-hot
conf         = 1   (this bin has a vertex)
```

Below we feed the square through the **real** `resample_polygons` + `_preprocess_polygons_v2` and print the resulting
`[24,3]` table — then hand‑check the four corner bins.

In [ ]:
# === Worked numeric example: a square, real encoder vs hand calculation ==================
cx_, cy_, hw_ = 0.5, 0.5, 0.15
corners = np.array([[cx_-hw_, cy_-hw_],[cx_+hw_, cy_-hw_],[cx_+hw_, cy_+hw_],[cx_-hw_, cy_+hw_]], np.float32)
sq_poly = corners.reshape(1, -1)
sq_box  = np.array([cy_-hw_, cx_-hw_, cy_+hw_, cx_+hw_], np.float32)     # yxyx
R = radial_encode(sq_box, sq_poly[0])                                    # REAL [24,3]
print("REAL encoder output  [24,3] = (dist, angle_offset, conf):  (occupied bins only)")
print(f"{'bin i':>5} {'angle°range':>12} {'dist':>7} {'offset':>7} {'conf':>5}")
for i in range(24):
    d,o,c = R[i]
    if c >= 0.5: print(f"{i:>5} {f'{i*15}-{(i+1)*15}':>12} {d:>7.3f} {o:>7.3f} {c:>5.0f}")
print("\nHAND-CHECK the 4 corners (relative to centre 0.5,0.5):")
for x,y in corners:
    dx,dy = x-cx_, y-cy_
    ang = math.degrees(math.atan2(dy,dx)) % 360
    print(f"  corner ({x:.2f},{y:.2f}):  radius={math.hypot(dx,dy):.3f}  angle={ang:6.1f}°  -> bin {int(ang//15)}  offset={(ang%15)/15:.3f}")
print("\n→ corner radii (0.212) match the encoder's dist in the corner bins; arc-resampling fills the edge bins too.")

REAL encoder output  [24,3] = (dist, angle_offset, conf):  (occupied bins only)
bin i  angle°range    dist  offset  conf
    0         0-15   0.155   0.936     1
    1        15-30   0.168   0.771     1
    2        30-45   0.199   0.746     1
    3        45-60   0.212   0.000     1
    4        60-75   0.168   0.229     1
    5        75-90   0.155   0.064     1
    6       90-105   0.155   0.936     1
    7      105-120   0.168   0.771     1
    8      120-135   0.199   0.746     1
    9      135-150   0.212   0.000     1
   10      150-165   0.168   0.229     1
   11      165-180   0.155   0.064     1
   12      180-195   0.155   0.936     1
   13      195-210   0.168   0.771     1
   14      210-225   0.199   0.746     1
   15      225-240   0.212   0.000     1
   16      240-255   0.168   0.229     1
   17      255-270   0.155   0.064     1
   18      270-285   0.155   0.936     1
   19      285-300   0.168   0.771     1
   20      300-315   0.199   0.746     1
   21      315-330

In [ ]:
# === Zoom into ONE 15° bin: what the sub-bin offset means ===============================
fig, ax = plt.subplots(figsize=(6.2, 6.2)); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Sub-bin angle offset (one 15° bin)", fontsize=12, weight="bold")
i0 = 1; a0, a1 = i0*15, (i0+1)*15           # bin 1: 15°..30°
# draw the wedge and neighbours
for k in range(0,3):
    aa = math.radians((i0-1+k)*15)
    ax.plot([0, 1.15*math.cos(aa)], [0, 1.15*math.sin(aa)], color="#bbb", lw=1)
th = np.radians(np.linspace(a0, a1, 30)); ax.fill(np.r_[0,np.cos(th)], np.r_[0,np.sin(th)], color="#cfe0f5", alpha=0.7)
# a vertex inside the bin
vang = 22.0; vr = 0.9; vx, vy = vr*math.cos(math.radians(vang)), vr*math.sin(math.radians(vang))
ax.plot([0,vx],[0,vy], color="#ff3b30", lw=2.2); ax.plot(vx,vy,'o', color="#ff3b30", ms=9)
ax.plot(0,0,'k+', ms=14, mew=2)
ax.annotate("bin start = %d°"%a0, (0.55*math.cos(math.radians(a0)),0.55*math.sin(math.radians(a0))), fontsize=9, color="#2f5d9e")
ax.annotate("bin end = %d°"%a1, (0.62*math.cos(math.radians(a1)),0.62*math.sin(math.radians(a1))), fontsize=9, color="#2f5d9e")
ax.annotate("vertex @ %.0f°\nradius = dist"%vang, (vx+0.03,vy), fontsize=9, color="#ff3b30")
ax.text(0.30,-0.16,"offset = (22−15)/15 = 0.47", fontsize=10, color="#9b2d5e", weight="bold")
ax.text(0.0,-0.32,"decode:  vertex_angle = (i + offset)·15° = (1 + 0.47)·15 = 22°", fontsize=9, color="#333")
ax.set_xlim(-0.2,1.2); ax.set_ylim(-0.45,1.1)
plt.tight_layout()
_save('03_subbin'); plt.show()

![03_subbin](assets/explainer/03_subbin.png)

<sub>The angle channel is a continuous offset *inside* the 15° bin — so 24 bins pin angles precisely, not coarsely.</sub>

### 3.2 The same encoding on a real object, step by step

In [ ]:
# === Visualise the radial encoding on a real object (…575), REAL encoder ================
ex = SAMPLES[2]; j = int(np.argmax((ex['boxes'][:,2]-ex['boxes'][:,0])*(ex['boxes'][:,3]-ex['boxes'][:,1])))
box = ex['boxes'][j]; poly = ex['polys'][j]; H=W=672
radial = radial_encode(box, poly); recon = radial_decode(box, radial)
cy = (box[0]+box[2])/2; cx = (box[1]+box[3])/2
raw_xy = poly.reshape(-1,2); raw_xy = raw_xy[raw_xy[:,0] > -1.0]
res = resample_polygons(tf.constant(poly[None,:],tf.float32), 64).numpy().reshape(-1,2); res = res[res[:,0] > -1.0]
fig, ax = plt.subplots(1, 4, figsize=(17, 4.6))
for a in ax: a.imshow(ex['image']); a.set_xticks([]); a.set_yticks([])
ax[0].plot(np.append(raw_xy[:,0],raw_xy[0,0])*W, np.append(raw_xy[:,1],raw_xy[0,1])*H, '-o', c='#34c759', ms=3, lw=1.2)
ax[0].set_title(f"1) raw polygon GT\n{raw_xy.shape[0]} stored vertices", fontsize=10)
ax[1].plot(np.append(res[:,0],res[0,0])*W, np.append(res[:,1],res[0,1])*H, '-o', c='#0a84ff', ms=3, lw=1.0)
ax[1].set_title("2) arc-length resample → 64 pts", fontsize=10)
ax[2].plot(cx*W, cy*H, 'r+', ms=14, mew=2)
for i in range(24):
    d, off, conf = radial[i]; th = math.radians((i+off)*15); vx, vy = cx + d*math.cos(th), cy + d*math.sin(th)
    col = '#ff3b30' if conf >= 0.4 else '#ccc'; ax[2].plot([cx*W, vx*W], [cy*H, vy*H], '-', c=col, lw=1.0)
    if conf >= 0.4: ax[2].plot(vx*W, vy*H, 'o', c='#ff9500', ms=3)
ax[2].set_title(f"3) 24 radial bins (15° each)\n{int((radial[:,2]>=0.4).sum())}/24 occupied", fontsize=10)
ax[3].add_patch(MplPolygon(np.stack([recon[:,0]*W, recon[:,1]*H],1), closed=True, fill=True, fc='#34c759', ec='#0a7d2c', lw=1.6, alpha=0.35))
ax[3].set_title(f"4) decoded back to cartesian\nθᵢ=(i+offset)·15°  ({recon.shape[0]} verts)", fontsize=10)
plt.tight_layout()
_save('03_radial_steps'); plt.show()

![03_radial_steps](assets/explainer/03_radial_steps.png)

<sub>raw vertices → arc-resample to 64 → 24 radial bins → decode back. The encode/decode round-trip is faithful.</sub>

In [ ]:
# === The [24,3] radial target itself ====================================================
fig, ax = plt.subplots(1, 3, figsize=(16, 3.0)); bins = np.arange(24)
ax[0].bar(bins, radial[:,0], color='#0a84ff'); ax[0].set_title("dist (radius per bin, normalised)")
ax[1].bar(bins, radial[:,1], color='#ff9500'); ax[1].set_title("angle (sub-bin offset ∈ [0,1))"); ax[1].set_ylim(0,1)
ax[2].bar(bins, radial[:,2], color='#34c759'); ax[2].set_title("conf (1 = bin has a vertex)"); ax[2].set_ylim(0,1.1)
for a in ax: a.set_xlabel("angular bin i  (i·15°)"); a.set_xticks([0,6,12,18,23])
plt.suptitle("PolyYOLO radial target [24,3] — the three channels the head learns to predict", fontsize=11, weight='bold')
plt.tight_layout()
_save('03_target_bars'); plt.show()

![03_target_bars](assets/explainer/03_target_bars.png)

<sub>The fixed-size target. `poly_dist`, `poly_angle`, `poly_conf` heads regress these three rows.</sub>

### 3.3 Why arc‑length resampling — the rectangle edge case
A 4‑corner **rectangle** would occupy only ~4 angular bins (the corner directions) if we kept its stored vertices —
a **diamond**, with the long flat edges left at `conf=0` (no gradient → "spiky" artifacts). The pipeline therefore
**arc‑length resamples** every contour to 64 points *uniformly along the boundary* (interpolating on edges), so
every bin the boundary crosses gets a vertex.

In [ ]:
# === Rectangle: stored-vertices vs arc-resampled (REAL encoder) =========================
exr = SAMPLES[0]; jr = int(np.argmax((exr['boxes'][:,2]-exr['boxes'][:,0])*(exr['boxes'][:,3]-exr['boxes'][:,1])))
boxr = exr['boxes'][jr]; polyr = exr['polys'][jr]
radial_raw = _PARSER._preprocess_polygons_v2(tf.constant(boxr[None,:],tf.float32), tf.constant(polyr[None,:],tf.float32), 15).numpy().reshape(-1,3)
radial_arc = radial_encode(boxr, polyr, resample=64)
rec_raw = radial_decode(boxr, radial_raw); rec_arc = radial_decode(boxr, radial_arc)
nverts = int((polyr.reshape(-1,2)[:,0] > -1.0).sum())
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a in ax: a.imshow(exr['image']); a.set_xticks([]); a.set_yticks([])
ax[0].add_patch(MplPolygon(np.stack([rec_raw[:,0]*672, rec_raw[:,1]*672],1), closed=True, fill=True, fc='#ff3b30', ec='#a01', lw=1.6, alpha=0.35))
ax[0].set_title(f"stored {nverts} vertices → {int((radial_raw[:,2]>=0.4).sum())}/24 bins\n(diamond — edges lost)", fontsize=10)
ax[1].add_patch(MplPolygon(np.stack([rec_arc[:,0]*672, rec_arc[:,1]*672],1), closed=True, fill=True, fc='#34c759', ec='#0a7d2c', lw=1.6, alpha=0.35))
ax[1].set_title(f"arc-resampled 64 → {int((radial_arc[:,2]>=0.4).sum())}/24 bins\n(true rectangle)", fontsize=10)
plt.suptitle("Arc-length resampling fills every bin the boundary crosses (parser.resample_points: 64)", fontsize=11, weight='bold')
plt.tight_layout()
_save('03_rectangle'); plt.show()

![03_rectangle](assets/explainer/03_rectangle.png)

<sub>Without arc-resampling a rectangle becomes a diamond; with it, all 24 bins are filled.</sub>

### 3.4 The format across many shapes — and its one limitation
The radial format keeps **one vertex per direction: the farthest one**. That is exact for **convex / star‑convex**
shapes (rectangles, blobs, most objects), but for a strongly **concave** boundary where a ray crosses the contour
more than once, only the outer crossing is kept — the dent is smoothed over. Below we encode→decode several real
objects (a rectangle, an organic shape, a thin object, a dense contour) and show where the format is faithful and
where it approximates.

In [ ]:
# === Gallery: encode→decode reconstruction over diverse real shapes =====================
picks = [(0,"rectangle (tv)"),(9,"organic (elephant)"),(4,"thin (bicycle)"),(5,"complex (motorcycle)")]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
for ax,(si,lab) in zip(axes, picks):
    ex = SAMPLES[si]; areas=(ex['boxes'][:,2]-ex['boxes'][:,0])*(ex['boxes'][:,3]-ex['boxes'][:,1]); j=int(np.argmax(areas))
    box=ex['boxes'][j]; poly=ex['polys'][j]; R=radial_encode(box,poly); rec=radial_decode(box,R)
    raw=poly.reshape(-1,2); raw=raw[raw[:,0]>-1.0]
    ax.imshow(ex['image']); ax.set_xticks([]); ax.set_yticks([])
    ax.plot(np.append(raw[:,0],raw[0,0])*672, np.append(raw[:,1],raw[0,1])*672, '-', c='#0a84ff', lw=1.4, label='GT')
    if len(rec): ax.add_patch(MplPolygon(np.stack([rec[:,0]*672,rec[:,1]*672],1), closed=True, fill=True, fc='#34c759', ec='#0a7d2c', lw=1.5, alpha=0.30))
    ax.set_title(f"{lab}\n{int((R[:,2]>=0.4).sum())}/24 bins", fontsize=9.5)
    ax.legend(loc='lower right', fontsize=7)
fig.suptitle("Encode→decode over diverse shapes (blue = GT contour, green = radial reconstruction)", fontsize=11, weight='bold')
plt.tight_layout()
_save('03_gallery'); plt.show()

![03_gallery](assets/explainer/03_gallery.png)

<sub>Faithful for convex/organic shapes; thin/concave structures are approximated (one farthest vertex per direction).</sub>

**Decode & the confidence gate (recap).** At inference/eval a bin contributes a vertex only when its **conf ≥ 0.4**;
the vertex sits at `θᵢ = (i + angle)·15°`, radius `dist`. Absent bins encode `dist=0, angle=0, conf=0`, so the
distance head learns to *collapse* non‑existent vertices. Validity throughout the pipeline is the **`-1.0` sentinel**
(`x > -1.0`, not `≥ 0`) so a mosaic‑overflow vertex with a slightly negative coordinate still counts
(`docs/design_register.md` entry 10). The conf channel is trained over **all 24 bins** (empty → 0) — the 2026‑06‑11
fix that removed spiky polygons (more in §9).

## 4 · Augmentations & how data flows

The training stream is built in `data_pipeline/input_reader.py:_build_detection_dataset`. The **stage order is
deliberate** — copy‑paste edits *within* one image, so it runs before mosaic stitches four images; geometry runs on
cheap `uint8` on the CPU, while colour augmentation moved to the **GPU inside `train_step`** to cut host→device
traffic 4×.

```
TFDS (SkipDecoding: images stay ENCODED through shuffle)
  → repeat() each source → sample_from_datasets(weights=[95,2,3])
  → shuffle(1500, seed=s)  → decode  → pre-resize 672²
  → zip(copy-paste source) → Copy-Paste (p=0.2)        # BEFORE mosaic
  → padded_batch(4, polygons pad = -1.0 sentinel)
  → Mosaic (4-in / 4-out, p=0.5) + random_perspective
  → unbatch → shuffle(128, seed=s+2)                   # decorrelate the 4-groups
  → parser: flip + polygon → radial [N,72]  (emits uint8)
  → batch(global_batch_size) → prefetch(AUTOTUNE)
  [train_step on GPU] HSV + Albumentations + /255
```

In [ ]:
# === CPU (tf.data) vs GPU (train_step) work split ======================================
fig, ax = plt.subplots(figsize=(13.5, 4.2)); ax.set_xlim(0,100); ax.set_ylim(0,30); ax.axis('off')
ax.add_patch(FancyBboxPatch((0.5,16),99,12.5, boxstyle="round,pad=0.1", fc="#fbf7ef", ec="#b6791f", lw=1.3))
ax.text(2,26.6,"CPU · tf.data pipeline (uint8, geometry + labels)", color="#b6791f", fontsize=11, weight='bold')
for i,s in enumerate(["decode","pre-resize\n672²","Copy-Paste\np=0.2","Mosaic 4→4\n+perspective","flip","poly→radial\n[N,72]","uint8 batch"]):
    node(ax, 2+i*13.6, 17.5, 12, 6.5, s, kind="block", fs=8)
    if i: arrow(ax, 2+i*13.6-1.6, 20.7, 2+i*13.6, 20.7)
ax.add_patch(FancyBboxPatch((52,1.5),47,10, boxstyle="round,pad=0.1", fc="#eef9f1", ec="#2e8b57", lw=1.3))
ax.text(53.5,9.6,"GPU · inside train_step", color="#2e8b57", fontsize=11, weight='bold')
for i,s in enumerate(["/255 normalize","HSV jitter","Albumentations\n(det rows only)"]):
    node(ax, 54+i*15, 2.5, 13.5, 5.5, s, kind="op", fs=8)
    if i: arrow(ax, 54+i*15-1.5, 5.2, 54+i*15, 5.2)
arrow(ax, 90, 17.5, 75, 8.2, "uint8 → GPU")
plt.tight_layout()
_save('04_cpu_gpu'); plt.show()

![04_cpu_gpu](assets/explainer/04_cpu_gpu.png)

<sub>Geometry stays on the CPU in uint8; colour augmentation runs on the GPU per batch.</sub>

In [ ]:
# === Adapters to feed the REAL augmentation modules (format glue only) ==================
def make_mosaic_batch(samples4):
    maxN = max(s['boxes'].shape[0] for s in samples4); maxF = max(s['polys'].shape[1] for s in samples4)
    def padN(a, val, N): out = np.full((N,)+a.shape[1:], val, a.dtype); out[:a.shape[0]] = a; return out
    polys = np.stack([padN(np.pad(s['polys'], ((0,0),(0,maxF-s['polys'].shape[1])), constant_values=-1.0), -1.0, maxN) for s in samples4])
    z = np.zeros((4,maxN), np.float32)
    return {"image": tf.constant(np.stack([s['image'] for s in samples4])),
            "groundtruth_boxes": tf.constant(np.stack([padN(s['boxes'],0.0,maxN) for s in samples4])),
            "groundtruth_classes": tf.constant(np.stack([padN(s['classes'],0,maxN) for s in samples4])),
            "groundtruth_polygons": tf.constant(polys), "groundtruth_is_crowd": tf.constant(np.zeros((4,maxN),bool)),
            "groundtruth_area": tf.constant(z), "groundtruth_dontcare": tf.constant(np.zeros((4,maxN),np.int64)),
            "groundtruth_dists": tf.constant(z), "source_id": tf.constant(["a","b","c","d"]),
            "height": tf.constant([672]*4,tf.int32), "width": tf.constant([672]*4,tf.int32)}
def make_cp_source(ex, j):
    H,W = ex['image'].shape[:2]; b = ex['boxes'][j]
    y1,x1,y2,x2 = (b*[H,W,H,W]).astype(int); y1,x1=max(0,y1),max(0,x1); y2,x2=min(H,y2),min(W,x2)
    crop = ex['image'][y1:y2, x1:x2]; ch,cw = crop.shape[:2]
    poly = ex['polys'][j].reshape(-1,2); poly = poly[poly[:,0] > -1.0]
    px = poly[:,0]*W - x1; py = poly[:,1]*H - y1
    mask = Image.new("L",(cw,ch),0); ImageDraw.Draw(mask).polygon(list(zip(px,py)), fill=255)
    rgba = np.dstack([crop, np.asarray(mask)[...,None]]).astype(np.uint8)
    pts = np.stack([px/max(cw,1), py/max(ch,1)],1).reshape(-1).astype(np.float32)
    return {"image": tf.constant(rgba), "orig_bbox": tf.constant([0.,0.,1.,1.],tf.float32),
            "label": tf.constant(ex['classes'][j],tf.int64), "points": tf.constant(pts)}
def full_dict(ex):
    n = ex['boxes'].shape[0]
    return {"image": tf.constant(ex['image']), "groundtruth_boxes": tf.constant(ex['boxes']),
            "groundtruth_classes": tf.constant(ex['classes']), "groundtruth_polygons": tf.constant(ex['polys']),
            "groundtruth_is_crowd": tf.constant(np.zeros(n,bool)), "groundtruth_area": tf.constant(np.zeros(n,np.float32)),
            "groundtruth_dontcare": tf.constant(np.zeros(n,np.int64)), "groundtruth_dists": tf.constant(np.full(n,-1.0,np.float32)),
            "height": tf.constant(672,tf.int32), "width": tf.constant(672,tf.int32)}
print("aug adapters ready")

aug adapters ready


### 4.1 Copy‑Paste (`copy_paste.py`, p=0.2, before mosaic) · 4.2 Mosaic (`mosaic.py`, 4‑in/4‑out, p=0.5)

In [ ]:
# === Copy-Paste (REAL module, forced) ===================================================
from data_pipeline.copy_paste import CopyAndPasteModule
bg = full_dict(SAMPLES[3]); src = make_cp_source(SAMPLES[2], int(np.argmax((SAMPLES[2]['boxes'][:,2]-SAMPLES[2]['boxes'][:,0]))))
cp = CopyAndPasteModule(prob=1.0).process_fn(is_training=True)(dict(bg), src)
fig, ax = plt.subplots(1, 2, figsize=(10.5, 5.2))
for a,(t,d) in zip(ax, [("before", bg), ("after copy-paste (+object)", cp)]):
    a.imshow(np_(d['image'])); a.set_xticks([]); a.set_yticks([])
    draw_boxes(a, np_(d['groundtruth_boxes'])); draw_polys(a, np_(d['groundtruth_polygons'])); a.set_title(t, fontsize=10)
plt.tight_layout()
_save('04_copypaste'); plt.show()

![04_copypaste](assets/explainer/04_copypaste.png)

<sub>Copy-Paste composites an RGBA object (alpha mask) onto the background and appends its box+polygon.</sub>

In [ ]:
# === Mosaic (REAL module, forced mosaic branch) =========================================
from data_pipeline.mosaic import Mosaic
four = [SAMPLES[i] for i in (4, 3, 6, 9)]; mb = make_mosaic_batch(four)
mo = Mosaic(output_size=[672,672], mosaic_frequency=1.0, with_polygons=True).mosaic_fn(True)(mb)
fig, ax = plt.subplots(1, 4, figsize=(17, 4.4))
for k in range(4):
    a = ax[k]; a.imshow(np_(mo['image'][k])); a.set_xticks([]); a.set_yticks([])
    draw_polys(a, np_(mo['groundtruth_polygons'][k])); draw_boxes(a, np_(mo['groundtruth_boxes'][k]))
    a.set_title(f"mosaic {k}  ·  {int((np_(mo['groundtruth_boxes'][k]).sum(1)!=0).sum())} boxes", fontsize=9)
fig.suptitle("Mosaic: 4 inputs → 4 outputs (rotated quadrant permutations), one perspective warp each", fontsize=11, weight='bold')
plt.tight_layout()
_save('04_mosaic'); plt.show()

![04_mosaic](assets/explainer/04_mosaic.png)

<sub>A 4-image group yields 4 mosaics; each input lands in each quadrant once. Boxes/polygons ride the warp.</sub>

### 4.3 Random perspective · 4.4 Horizontal flip (`augmentations.py`) · 4.5 Colour aug (`batch_color_aug.py`, GPU)

In [ ]:
# === Perspective + flip (REAL) ==========================================================
from data_pipeline.augmentations import random_perspective, random_horizontal_flip
exg = SAMPLES[4]
io, bo, keep, po = random_perspective(tf.constant(exg['image']), tf.constant(exg['boxes']), tf.constant(exg['polys']), 672, 672, scale_min=0.4, scale_max=1.9)
tf.random.set_seed(4); fi, fb, fp = random_horizontal_flip(tf.constant(exg['image']), tf.constant(exg['boxes']), tf.constant(exg['polys']))
fig, ax = plt.subplots(1, 3, figsize=(14.5, 5))
for a,(t,im,bx,pl) in zip(ax, [("original", exg['image'], exg['boxes'], exg['polys']),
        ("random_perspective", np_(io), np_(bo), np_(po)), ("horizontal flip", np_(fi), np_(fb), np_(fp))]):
    a.imshow(im); a.set_xticks([]); a.set_yticks([]); draw_polys(a, pl); draw_boxes(a, bx); a.set_title(t, fontsize=10)
plt.tight_layout()
_save('04_perspective_flip'); plt.show()

![04_perspective_flip](assets/explainer/04_perspective_flip.png)

<sub>random_perspective composes centre→perspective→rotate·scale→shear→translate; flip mirrors x↔1−x.</sub>

In [ ]:
# === Colour augmentation (REAL GPU path) ================================================
from data_pipeline.batch_color_aug import batch_color_augment
imgs = tf.stack([tf.constant(s['image']) for s in SAMPLES]); tf.random.set_seed(7)
caug = batch_color_augment(imgs, hue=0.015, sat=0.7, val=0.4, albu_freq=1.0, albu_row_mask=tf.ones([10], tf.bool))
fig, axes = plt.subplots(2, 5, figsize=(16.5, 7))
for k, a in enumerate(axes.ravel()):
    a.imshow(np_(caug[k])); a.set_xticks([]); a.set_yticks([]); a.set_title(f"…{SAMPLES[k]['stem'][-3:]}", fontsize=8)
fig.suptitle("Colour-augmented batch (HSV + Albumentations, /255) — geometry & labels unchanged", fontsize=12, weight='bold')
plt.tight_layout()
_save('04_colour'); plt.show()

![04_colour](assets/explainer/04_colour.png)

<sub>HSV jitter + Albumentations applied per image on the GPU; only detection rows get Albumentations.</sub>

### 4.6 Two streams, merged at the batch level
Training merges a **detection** stream (128/batch, box+polygon GT, `ignore_bg=0`) with a **distance** stream
(16/batch, `servingbot_polygon`, real distance GT, `ignore_bg=1` → class loss masked to foreground, skips
Albumentations; **training‑only**) via `tf.data.zip()` + batch‑axis concat → **144** rows/step. The infinite stream
runs exactly `steps_per_loop = 271,166 // 128 = 2118` steps per epoch. Three shuffles use distinct seeds `s, s+1, s+2`.

In [ ]:
# === Two-stream merge =================================================================
fig, ax = plt.subplots(figsize=(12.5, 4.0)); ax.set_xlim(0,100); ax.set_ylim(0,26); ax.axis('off')
node(ax, 1, 16, 26, 7.5, "DETECTION stream", "box+polygon GT · ignore_bg=0 · batch 128", "block", fs=9)
node(ax, 1, 3, 26, 7.5, "DISTANCE stream", "distance GT · ignore_bg=1 · batch 16 · train-only", "concat", fs=9)
node(ax, 38, 9.5, 20, 8, "tf.data.zip()", "+ concat on\nbatch axis", "op", fs=9.5)
node(ax, 66, 9.5, 30, 8, "merged batch = 144", "→ model → loss\n(dist rows: cls→fg, no poly loss)", "out", fs=9)
arrow(ax, 27, 19.5, 38, 14.5); arrow(ax, 27, 6.7, 38, 12); arrow(ax, 58, 13.5, 66, 13.5)
plt.tight_layout()
_save('04_two_streams'); plt.show()

![04_two_streams](assets/explainer/04_two_streams.png)

<sub>Detection + distance streams zip together; distance-only rows mask the classification loss.</sub>

## 5 · The model — intuition first, then every component

Three stages turn a `672²×3` image into per‑cell predictions. The intuition for *why* there are three:

- **BACKBONE — "what is in the image, at many scales."** A stack of convolutions that repeatedly **halves the
  resolution and doubles the channels**. Early layers see edges/texture at high resolution; deep layers see whole
  objects at low resolution. It emits feature maps at **three scales** (strides 8/16/32) so small *and* large objects
  are both representable.
- **DECODER (FPN‑PAN) — "mix fine detail with global meaning."** Deep features know *what* but are coarse; shallow
  features know *where* but lack context. The decoder fuses them **top‑down** (spread semantics to fine levels) then
  **bottom‑up** (send precise localization back up) so every output level is both detailed and semantic.
- **HEAD — "decide, per cell."** Lightweight conv branches that, at every cell of every level, predict a box, a
  class, a polygon and a distance.

Why three scales at all? An object only gets good predictions from the level whose cells are the right size for it:

> **In plain words — a three‑stage factory.** Think of the model as an assembly line. The **backbone** is the
> *inspector* who looks at the raw photo and writes increasingly abstract notes ("edge here → wheel → bicycle"),
> zooming out at each step. The **decoder** is a *committee* that passes those notes up and down the line so a note
> about fine detail and a note about the big picture end up on the same desk. The **head** is the *team of clerks*
> who, looking at one small patch each, fill in a form: *is there an object here? what class? what shape? how far?*

In [ ]:
# === Stride / scale intuition: 3 grids over an image ===================================
ex = SAMPLES[7]                                  # busy dining scene
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, st, col, lab in zip(ax, [8,16,32], ["#34c759","#0a84ff","#ff3b30"], ["small","medium","large"]):
    a.imshow(ex['image']); a.set_xticks([]); a.set_yticks([]); n = 672//st
    for k in range(0, 672, st*8):                # draw every 8th gridline (else too dense)
        a.axhline(k, color=col, lw=0.5, alpha=0.5); a.axvline(k, color=col, lw=0.5, alpha=0.5)
    a.add_patch(Rectangle((40,40), st, st, fc=col, ec='k'));
    a.set_title(f"stride {st}  →  {n}×{n} cells\none cell = {st}×{st}px  (detects {lab} objects)", fontsize=10)
fig.suptitle("Anchor-free: one prediction per cell. Fine grid (stride 8) for small objects, coarse (stride 32) for large.",
             fontsize=11, weight='bold'); plt.tight_layout()
_save('05_strides'); plt.show()

![05_strides](assets/explainer/05_strides.png)

<sub>Each level's cell size suits a different object scale — that is why the model predicts at 3 strides.</sub>

In [ ]:
# === Build the REAL model and read every shape =========================================
cfg.task.model.deploy = False
model = build_yolov8(cfg.task.model)
x = tf.zeros([1, 672, 672, 3]); _ = model(x, training=False)
feats_bb = model.backbone(x, training=False); feats_dec = model.decoder(feats_bb, training=False); out = model.head(feats_dec, training=False)
print("BACKBONE outputs:", {l: tuple(feats_bb[l].shape) for l in '345'}, "(strides 8/16/32)")
print("DECODER  outputs:", {l: tuple(feats_dec[l].shape) for l in '345'})
print("\nHEAD outputs (6 branches × 3 levels):")
for h in out: print(f"   {h:12} ch={out[h]['3'].shape[-1]:>3}  " + "  ".join(f"P{l}:{tuple(out[h][l].shape)}" for l in '345'))
print(f"\nTotal params: {model.count_params():,}   ·   anchors 84²+42²+21² = {84*84+42*42+21*21}")

BACKBONE outputs: {'3': (1, 84, 84, 128), '4': (1, 42, 42, 256), '5': (1, 21, 21, 512)} (strides 8/16/32)
DECODER  outputs: {'3': (1, 84, 84, 128), '4': (1, 42, 42, 256), '5': (1, 21, 21, 512)}

HEAD outputs (6 branches × 3 levels):
   box          ch= 64  P3:(1, 84, 84, 64)  P4:(1, 42, 42, 64)  P5:(1, 21, 21, 64)
   cls          ch= 39  P3:(1, 84, 84, 39)  P4:(1, 42, 42, 39)  P5:(1, 21, 21, 39)
   poly_angle   ch= 24  P3:(1, 84, 84, 24)  P4:(1, 42, 42, 24)  P5:(1, 21, 21, 24)
   poly_dist    ch= 24  P3:(1, 84, 84, 24)  P4:(1, 42, 42, 24)  P5:(1, 21, 21, 24)
   poly_conf    ch= 24  P3:(1, 84, 84, 24)  P4:(1, 42, 42, 24)  P5:(1, 21, 21, 24)
   dist         ch=  1  P3:(1, 84, 84, 1)  P4:(1, 42, 42, 1)  P5:(1, 21, 21, 1)

Total params: 13,219,376   ·   anchors 84²+42²+21² = 9261


In [ ]:
# === Backbone diagram with REAL shapes (feature-map slabs) =============================
ns=[max(1,round(n*0.33)) for n in [3,6,6,3]]; P=[feats_bb[l].shape[-1] for l in '345']
fig, ax = plt.subplots(figsize=(15, 5.4)); ax.set_xlim(0,104); ax.set_ylim(0,40); ax.axis('off')
ax.set_title("BACKBONE · CSPDarkNetV8-S  (width 0.5, depth 0.33)  — resolution halves, channels grow", fontsize=12, weight='bold')
specs=[("input","672²×3",16,"input",None),("stem","168²×64",12,"conv",None),
       ("P3","84²×%d"%P[0],10,"block",8),("P4","42²×%d"%P[1],8,"block",16),("P5 +SPPF","21²×%d"%P[2],6,"block",32)]
xs=[4,26,46,64,80]
for (name,shp,sz,kind,st),xx in zip(specs,xs):
    slab(ax, xx, 20-sz/2, sz, sz, "", kind=kind, d=2.2)
    ax.text(xx+sz/2, 20+sz/2+4.5, name, ha='center', fontsize=9.5, weight='bold', color="#11223a")
    ax.text(xx+sz/2, 20-sz/2-3.0, shp, ha='center', fontsize=8.5, color="#33475e")
    if st: ax.text(xx+sz/2, 20-sz/2-6.0, "stride %d"%st, ha='center', fontsize=7.5, color="#9b2d5e")
for a,b in zip(xs[:-1],xs[1:]): arrow(ax, a+ {4:16,26:12,46:10,64:8}[a] +1.5, 20, b-1.5, 20)
ax.text(52, 1, "C2f repeats per stage = round(base·0.33) = %s   ·   the 3 emitted levels (P3/P4/P5) feed the decoder" % ns,
        fontsize=9, ha='center', color="#555")
plt.tight_layout()
_save('05_backbone'); plt.show()

![05_backbone](assets/explainer/05_backbone.png)

<sub>Real shapes: 672²×3 → 84²×128 / 42²×256 / 21²×512 at strides 8/16/32 — deeper layers are smaller and richer.</sub>

In [ ]:
# === C2f and SPPF internals with a concrete channel example (C=128) ====================
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
for a in ax: a.set_xlim(0,100); a.set_ylim(0,40); a.axis('off')
ax[0].set_title("C2f  (C=128, n=2 bottlenecks)  — split, process half, concat all", fontsize=11, weight='bold')
node(ax[0], 1, 16, 11, 7, "in", "H×W×128", "input", fs=8)
node(ax[0], 15, 16, 11, 7, "Conv1×1", "→128", "conv", fs=8); arrow(ax[0],12,19.5,15,19.5)
node(ax[0], 31, 25, 9, 6, "half a", "64", "pool", fs=8); node(ax[0], 31, 13, 9, 6, "half b", "64", "pool", fs=8)
arrow(ax[0],26,20,31,28); arrow(ax[0],26,19,31,16)
node(ax[0], 45, 6, 12, 6, "Bottleneck×2", "3×3,3×3 +short", "out", fs=7.5); arrow(ax[0],40,16,45,10)
node(ax[0], 64, 14, 11, 12, "concat", "4×64 = 256", "concat", fs=8.5)
arrow(ax[0],40,28,64,22); arrow(ax[0],40,16,64,19); arrow(ax[0],57,9,64,16)
node(ax[0], 80, 16, 12, 7, "Conv1×1", "→128 out", "conv", fs=8); arrow(ax[0],75,19.5,80,19.5)
ax[1].set_title("SPPF  — 3 chained 5×5 pools ≈ one big receptive field", fontsize=11, weight='bold')
node(ax[1], 1, 16, 10, 7, "in", "C", "input", fs=8)
node(ax[1], 14, 16, 10, 7, "Conv1×1", "→C/2", "conv", fs=8); arrow(ax[1],11,19.5,14,19.5)
for k,xx in enumerate([28,40,52]):
    node(ax[1], xx, 16, 10, 7, "MaxPool", "5×5 s1", "pool", fs=8); arrow(ax[1],xx-2,19.5,xx,19.5)
node(ax[1], 68, 14, 11, 11, "concat", "4×(C/2)\n= 2C", "concat", fs=8.5)
for xx in [24,38,50,62]: arrow(ax[1],xx,19,68,19.5)
node(ax[1], 84, 16, 11, 7, "Conv1×1", "→C", "conv", fs=8); arrow(ax[1],79,19.5,84,19.5)
plt.tight_layout()
_save('05_c2f_sppf'); plt.show()

![05_c2f_sppf](assets/explainer/05_c2f_sppf.png)

<sub>C2f keeps a gradient-friendly split/concat (CSP); SPPF stacks cheap 5×5 pools for multi-scale context.</sub>

In [ ]:
# === FPN-PAN decoder with REAL shapes ==================================================
c3,c4,c5 = [feats_bb[l].shape[-1] for l in '345']; d3,d4,d5 = [feats_dec[l].shape[-1] for l in '345']
fig, ax = plt.subplots(figsize=(13.5, 6.4)); ax.set_xlim(0,100); ax.set_ylim(0,54); ax.axis('off')
ax.set_title("DECODER · FPN (top-down semantics ↓) + PAN (bottom-up localization ↑)", fontsize=12, weight='bold')
node(ax, 2, 42, 15, 7, "P5", f"21²×{c5}", "block", fs=9); node(ax, 2, 27, 15, 7, "P4", f"42²×{c4}", "block", fs=9); node(ax, 2, 12, 15, 7, "P3", f"84²×{c3}", "block", fs=9)
node(ax, 33, 35, 19, 7, "up(P5)⊕P4 → C2f", f"{c5}+{c4} → {c4}", "up", fs=8.5)
node(ax, 33, 12, 19, 7, "up(P4')⊕P3 → C2f", f"{c4}+{c3} → {c3}", "up", fs=8.5)
arrow(ax, 17, 45, 33, 39, "upsample", lcol="#1f8a8a"); arrow(ax, 17, 30, 33, 38); arrow(ax, 17, 15, 33, 15); arrow(ax, 42, 35, 42, 19, "", lcol="#1f8a8a")
node(ax, 66, 12, 20, 7, "P3'  OUT", f"84²×{d3}", "out", fs=9)
node(ax, 66, 27, 20, 7, "down(P3')⊕P4'→C2f", f"42²×{d4}", "down", fs=8.5)
node(ax, 66, 42, 20, 7, "down(P4'')⊕P5→C2f", f"21²×{d5}", "down", fs=8.5)
arrow(ax, 52, 15, 66, 15); arrow(ax, 76, 19, 76, 27, "downsample", lcol="#c0651f"); arrow(ax, 52, 16, 66, 30); arrow(ax, 76, 34, 76, 42); arrow(ax, 42, 39, 66, 46)
ax.text(50, 3, "every output level now carries both fine detail (from P3) and global context (from P5)", fontsize=9, ha='center', color="#555")
plt.tight_layout()
_save('05_fpn_pan'); plt.show()

![05_fpn_pan](assets/explainer/05_fpn_pan.png)

<sub>FPN spreads deep semantics down; PAN sends precise localization back up. Concat channel math is annotated.</sub>

In [ ]:
# === Head: shared cv2feat stem + 6 branches ============================================
nc=cfg.task.model.num_classes; ps=cfg.task.model.output_poly_size; rm=16; cv2h=4*rm+3*ps
fig, ax = plt.subplots(figsize=(13.5, 5.8)); ax.set_xlim(0,100); ax.set_ylim(0,46); ax.axis('off')
ax.set_title(f"HEAD · per-pixel, all 3 levels  (num_classes={nc}, poly_size={ps}, reg_max={rm})", fontsize=12, weight='bold')
node(ax, 1, 19, 14, 8, "decoder", "feature\n(per level)", "block", fs=9)
node(ax, 20, 28, 16, 7, "cv2feat stem", f"Conv3×3 ×2 → {cv2h}", "conv", fs=8.5)
node(ax, 20, 11, 16, 7, "cls stem", "Conv3×3 ×2 → 128", "conv", fs=8.5)
node(ax, 20, 2, 16, 7, "dist stem", "Conv3×3 → 128", "conv", fs=8.5)
arrow(ax,15,23.5,20,31.5); arrow(ax,15,23,20,14.5); arrow(ax,15,22,20,5.5)
for t,sub,yy in [("box","4·reg_max = 64",38),("poly_angle","24",31.5),("poly_dist","24",25),("poly_conf","24",18.5)]:
    node(ax, 42, yy, 22, 5.6, t, sub, "concat", fs=8.5); arrow(ax, 36, 31.5, 42, yy+2.8)
node(ax, 42, 11, 22, 5.6, "cls", f"{nc}", "out", fs=8.5); arrow(ax, 36, 14.5, 42, 13.8)
node(ax, 42, 2, 22, 5.6, "dist", "1", "out", fs=8.5); arrow(ax, 36, 5.5, 42, 4.8)
ax.text(85, 22, "all 6 prediction\nConv2D pinned float32\n(under mixed_bfloat16);\nsmart-bias on cls & box",
        fontsize=8.5, ha='center', va='center', bbox=dict(boxstyle="round", fc="#fffbe8", ec="#b6791f"))
plt.tight_layout()
_save('05_head'); plt.show()

![05_head](assets/explainer/05_head.png)

<sub>A shared stem feeds box + 3 polygon branches; cls and dist get their own stems. 6 outputs per cell.</sub>

## 6 · How the heads change when the number of classes changes

Only **one** thing in the whole network depends on `num_classes`: the final **1×1 `cls_pred` convolution** at each
of the 3 levels (plus its smart‑bias). Backbone, decoder, the box/poly/dist branches and even the shared `cv2feat`
stem are **independent of `num_classes`**.

| Head conv | output channels | depends on |
|---|---|---|
| `cls_pred_{3,4,5}` | **`num_classes`** | **num_classes** |
| `box_pred` | `4·reg_max = 64` | reg_max |
| `poly_angle/dist/conf` | `output_poly_size = 24` | `360/angle_step` |
| `dist_pred` | `1` | — |
| shared `cv2feat` | `4·reg_max + 3·poly = 136` | reg_max, poly_size |

`smart‑bias` then sets `cls_bias = log(5 / num_classes / (input_size/stride)²)` — more classes ⇒ each starts less
confident. Let's prove it with the **real** `YoloV8Head` at three class counts.

In [ ]:
# === REAL YoloV8Head for num_classes ∈ {2,39,80}: what changed ==========================
from models.head import YoloV8Head
fp = {"3": tf.zeros([1,84,84,128]), "4": tf.zeros([1,42,42,256]), "5": tf.zeros([1,21,21,512])}
print(f"{'num_classes':>11} | {'cls_pred_3 kernel':>22} | {'box_pred_3':>14} | {'poly_angle_3':>14} | cls smart-bias(str8)")
print("-"*98)
for ncv in (2, 39, 80):
    h = YoloV8Head(num_classes=ncv, output_poly_size=24, output_dist_size=1, reg_max=16, smart_bias=True, with_polygons=True, with_distance=True)
    _ = h(fp);  h.initialize_biases(672) if hasattr(h,"initialize_biases") else None
    print(f"{ncv:>11} | {str(tuple(h.cls_pred_3.kernel.shape)):>22} | {str(tuple(h.box_pred_3.kernel.shape)):>14} | "
          f"{str(tuple(h.pa_pred_3.kernel.shape)):>14} | {math.log(5.0/ncv/(672/8)**2):+.3f}")
print("\n→ only cls_pred (and its bias) scales with num_classes; box/poly/cv2feat are identical across all three.")

num_classes |      cls_pred_3 kernel |     box_pred_3 |   poly_angle_3 | cls smart-bias(str8)
--------------------------------------------------------------------------------------------------
          2 |         (1, 1, 128, 2) | (1, 1, 136, 64) | (1, 1, 136, 24) | -7.945


         39 |        (1, 1, 128, 39) | (1, 1, 136, 64) | (1, 1, 136, 24) | -10.916
         80 |        (1, 1, 128, 80) | (1, 1, 136, 64) | (1, 1, 136, 24) | -11.634

→ only cls_pred (and its bias) scales with num_classes; box/poly/cv2feat are identical across all three.


In [ ]:
# === Config → head channel wiring ======================================================
fig, ax = plt.subplots(figsize=(12.5, 4.2)); ax.set_xlim(0,100); ax.set_ylim(0,30); ax.axis('off')
ax.set_title("Config → head channel wiring", fontsize=12, weight='bold')
for t,yy,ec in [("num_classes",24,"#9b2d5e"),("reg_max",15,"#b6791f"),("output_poly_size\n= 360/angle_step",6,"#2e8b57")]:
    ax.add_patch(FancyBboxPatch((2,yy),22,5, boxstyle="round,pad=0.02,rounding_size=0.05", fc="white", ec=ec, lw=1.6))
    ax.text(13,yy+2.5,t, ha='center', va='center', fontsize=9)
for t,fc,ec,yy in [("cls_pred → num_classes","#f6cfe0","#9b2d5e",24),("box_pred → 4·reg_max (64)","#ffe0b3","#b6791f",17),
        ("cv2feat → 4·reg_max+3·poly (136)","#e9f6ee","#2e8b57",10),("poly_angle/dist/conf → poly (24)","#c9efd6","#2e8b57",3)]:
    ax.add_patch(FancyBboxPatch((52,yy),44,5, boxstyle="round,pad=0.02,rounding_size=0.05", fc=fc, ec=ec, lw=1.4))
    ax.text(74,yy+2.5,t, ha='center', va='center', fontsize=9)
arrow(ax,24,26.5,52,26.5); arrow(ax,24,17.5,52,19.5); arrow(ax,24,17.5,52,12.5); arrow(ax,24,8.5,52,12); arrow(ax,24,8.5,52,5.5)
ax.text(50,0.2,"backbone + decoder + box/dist branches are independent of num_classes", fontsize=8.5, ha='center', color="#555")
plt.tight_layout()
_save('06_wiring'); plt.show()

![06_wiring](assets/explainer/06_wiring.png)

<sub>num_classes only resizes cls_pred; reg_max/poly_size drive box/poly channels. Everything else is fixed.</sub>

Everything downstream follows automatically: the cls loss is per‑class BCE over `C`
(`tal_loss.py:_class_loss`), the assigner builds `one_hot(label, C)` soft targets, and inference loops NMS over `C`
classes. **Change `num_classes` in the YAML and all of these resize consistently — no other code edits.**

## 7 · What the model predicts, and how to decode it — concretely

In **training mode** the model returns the **raw per‑pixel head outputs** (`head → level → tensor`). In **deploy
mode** the `detection_generator` decodes those into boxes/scores/polygons/distance + NMS.

**The anchor grid.** The model is **anchor‑free**: exactly one prediction per cell, and the cell at row `j`, col `i`
of a level owns the image point **`((i+0.5)·stride, (j+0.5)·stride)`** — its *anchor centre*. Box offsets are measured
from there. Across the 3 levels there are **84²+42²+21² = 9,261** cells.

Per cell, the raw outputs are:

| head | channels | raw meaning | decode |
|---|---|---|---|
| `box` | 64 = 4×16 | a 16‑bin **distribution** for each of L,T,R,B | softmax·bins → ltrb, ×stride |
| `cls` | 39 | per‑class logit | sigmoid |
| `poly_angle` | 24 | per‑bin sub‑bin offset logit | sigmoid → `[0,1)` |
| `poly_dist` | 24 | per‑bin radius logit | softplus |
| `poly_conf` | 24 | per‑bin validity logit | sigmoid (gate 0.4) |
| `dist` | 1 | log‑scale distance | exp, clip `[0.5,10]` m |

Let's take **one concrete cell** and decode its box by hand using the real `detection_generator`.

In [ ]:
# === One cell, fully decoded: raw box (64) → DFL → ltrb → xyxy (REAL detection_generator) ===
cfg.task.model.deploy = True; model_d = build_yolov8(cfg.task.model)
img = (tf.cast(tf.constant(SAMPLES[2]['image']), tf.float32)/255.0)[None]
raw = model(img, training=False)                                   # training-mode raw heads (deploy=False model)
print("RAW head outputs (level 3 shapes):", {h: tuple(raw[h]['3'].shape) for h in raw})
print("\nDECODED deploy outputs:")
det = model_d(img, training=False)
for k,v in det.items(): print(f"   {k:16} {tuple(v.shape)}  {v.dtype.name}")
# --- pick one cell on the coarse level (stride 32) and decode its box ---
lvl, st = '5', 32; jj, ii = 10, 10
boxraw = raw['box'][lvl]                                            # [1,21,21,64]
ltrb_fm = model_d.detection_generator._decode_dfl(tf.cast(boxraw, tf.float32))[0, jj, ii].numpy()  # REAL -> 4 (feature units)
acx, acy = (ii+0.5)*st, (jj+0.5)*st                                # anchor centre in pixels
l,t,r,b = ltrb_fm*st                                               # to pixels
x1,y1,x2,y2 = acx-l, acy-t, acx+r, acy+b
print(f"\ncell (row {jj}, col {ii}) on stride-{st} level:")
print(f"   anchor centre = ((i+0.5)·stride, (j+0.5)·stride) = ({acx:.0f}, {acy:.0f}) px")
print(f"   DFL-decoded ltrb (feature units) = {np.round(ltrb_fm,2)}  → ×{st} px = {np.round(ltrb_fm*st,1)}")
print(f"   box xyxy = (cx−l, cy−t, cx+r, cy+b) = ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}) px  (untrained → arbitrary)")

RAW head outputs (level 3 shapes): {'box': (1, 84, 84, 64), 'cls': (1, 84, 84, 39), 'poly_angle': (1, 84, 84, 24), 'poly_dist': (1, 84, 84, 24), 'poly_conf': (1, 84, 84, 24), 'dist': (1, 84, 84, 1)}

DECODED deploy outputs:


   bbox             (1, 300, 4)  float32
   classes          (1, 300)  int64
   confidence       (1, 300)  float32
   num_detections   (1,)  int32
   polygons         (1, 300, 24, 3)  float32
   distance         (1, 300)  float32

cell (row 10, col 10) on stride-32 level:
   anchor centre = ((i+0.5)·stride, (j+0.5)·stride) = (336, 336) px
   DFL-decoded ltrb (feature units) = [7.5 7.5 7.5 7.5]  → ×32 px = [240.  240.  240.1 240. ]
   box xyxy = (cx−l, cy−t, cx+r, cy+b) = (96, 96, 576, 576) px  (untrained → arbitrary)


In [ ]:
# === DFL decode visual + box reconstruction ============================================
logits4 = tf.reshape(boxraw[0,jj,ii], [4,16]); probs = tf.nn.softmax(tf.cast(logits4,tf.float32),axis=-1).numpy()
bins = np.arange(16); sides=['left','top','right','bottom']
fig = plt.figure(figsize=(15, 3.2));
for s in range(4):
    a = fig.add_subplot(1,5,s+1); a.bar(bins, probs[s], color='#0a84ff')
    ev = float((probs[s]*bins).sum()); a.axvline(ev, color='#ff3b30', lw=2)
    a.set_title(f"{sides[s]}: E={ev:.2f}", fontsize=9.5); a.set_xlabel("DFL bin")
a5 = fig.add_subplot(1,5,5); a5.set_xlim(0,1); a5.set_ylim(0,1); a5.axis('off'); a5.set_title("box reconstruction", fontsize=9.5)
a5.plot(0.5,0.5,'r+',ms=14,mew=2); a5.text(0.5,0.42,"anchor\ncentre",ha='center',fontsize=8,color='r')
a5.add_patch(Rectangle((0.22,0.25),0.56,0.5, fill=False, ec='#2e8b57', lw=2))
for (dx,dy,lab) in [(-0.28,0,"l"),(0.28,0,"r"),(0,0.25,"t"),(0,-0.25,"b")]:
    a5.annotate("", (0.5+dx,0.5+dy),(0.5,0.5), arrowprops=dict(arrowstyle="->",color='#9b2d5e'))
    a5.text(0.5+dx*1.25,0.5+dy*1.4,lab,color='#9b2d5e',fontsize=10,ha='center')
a5.text(0.5,0.06,"x1=cx−l·s  y1=cy−t·s\nx2=cx+r·s  y2=cy+b·s",ha='center',fontsize=7.5)
plt.suptitle("DFL: each box side = expected value of a 16-bin softmax (sub-cell precision), then ×stride", fontsize=11, weight='bold')
plt.tight_layout()
_save('07_dfl'); plt.show()

![07_dfl](assets/explainer/07_dfl.png)

<sub>Why a distribution? Predicting a softmax over bins (then its mean) gives smooth, sub-pixel box edges.</sub>

### 7.1 Polygon & distance decode
The polygon prediction is `predictions['polygons'] = [B, 300, 24, 3]` ordered **`(conf, dist, angle)`**, already
activated (conf/angle sigmoid, dist softplus). Decoding is exactly the radial→cartesian map from §3: keep bins with
`conf ≥ 0.4`, place a vertex at `θ = (i + angle)·15°`, radius `dist`. Distance is `exp` then clipped to `[0.5,10]` m.
Below we decode the **GT‑encoded** radial of a real object (clean, since the model is untrained) to show the
mechanism, and plot the distance map.

In [ ]:
# === Polygon decode (mechanism, on a clean radial) + distance decode ====================
ex = SAMPLES[9]; j = int(np.argmax((ex['boxes'][:,2]-ex['boxes'][:,0])*(ex['boxes'][:,3]-ex['boxes'][:,1])))
R = radial_encode(ex['boxes'][j], ex['polys'][j]); verts = radial_decode(ex['boxes'][j], R)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].imshow(ex['image']); ax[0].set_xticks([]); ax[0].set_yticks([])
ax[0].add_patch(MplPolygon(np.stack([verts[:,0]*672,verts[:,1]*672],1), closed=True, fill=True, fc='#34c759', ec='#0a7d2c', lw=1.6, alpha=0.4))
ax[0].set_title("polygon decode: keep conf≥0.4 → vertex at (i+angle)·15°, radius=dist", fontsize=9.5)
log_d = np.linspace(math.log(0.3), math.log(15), 200); metres = np.clip(np.exp(log_d), 0.5, 10.0)
ax[1].plot(log_d, metres, lw=2, color='#1f8a8a'); ax[1].axhline(0.5, color='#999', ls='--'); ax[1].axhline(10.0, color='#999', ls='--')
ax[1].set_xlabel("raw head output (log-scale)"); ax[1].set_ylabel("metres")
ax[1].set_title("distance decode: d = clip(exp(raw), 0.5, 10) m", fontsize=9.5)
plt.tight_layout()
_save('07_poly_dist_decode'); plt.show()

![07_poly_dist_decode](assets/explainer/07_poly_dist_decode.png)

<sub>Polygon: radial→cartesian via the conf gate. Distance: trained in log-space, decoded with exp+clip.</sub>

### 7.2 From 9,261 raw cells to a clean detection list — NMS
Most of the 9,261 cells fire weakly or duplicate a neighbour. The `detection_generator`
(`models/detection_generator.py`) decodes all cells, then runs **per‑class greedy NMS**
(`max_boxes=300`, `iou=0.65`, `score=0.05`): for each class independently it keeps the highest‑scoring box and
suppresses overlapping ones. The polygon/distance of each surviving box ride along.

In [ ]:
# === NMS intuition ====================================================================
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
rng = np.random.default_rng(0)
ax[0].set_title("before NMS: many overlapping candidate boxes", fontsize=10)
base = np.array([0.3,0.3,0.7,0.7])
for _ in range(22):
    j = base + rng.normal(0,0.035,4); ax[0].add_patch(Rectangle((j[0],j[1]),j[2]-j[0],j[3]-j[1], fill=False, ec='#ff3b30', lw=1, alpha=0.5))
ax[1].set_title("after NMS: one box per object per class", fontsize=10)
ax[1].add_patch(Rectangle((base[0],base[1]),base[2]-base[0],base[3]-base[1], fill=False, ec='#2e8b57', lw=2.4))
for a in ax: a.set_xlim(0,1); a.set_ylim(0,1); a.set_xticks([]); a.set_yticks([]); a.set_aspect('equal'); a.invert_yaxis()
plt.suptitle("Per-class greedy NMS — keep the best, suppress IoU>0.65 neighbours, cap at 300", fontsize=11, weight='bold')
plt.tight_layout()
_save('07_nms'); plt.show()

![07_nms](assets/explainer/07_nms.png)

<sub>NMS collapses thousands of overlapping raw predictions into one clean box per object, per class.</sub>

## 8 · Building targets — Task‑Aligned Assignment

The model emits **9,261 predictions** per image, but an image has only a few objects. **Label assignment** decides
which cells are *responsible* for which ground‑truth object — turning sparse GT into the dense target tensors the loss
compares against.

YOLOv8 uses **Task‑Aligned Assignment** (`losses/tal_assigner.py`). A cell is a good match for a GT when it does
**both** jobs well — predicts the right class (score) **and** localizes well (IoU) — captured by one metric:

$$\text{align} = \text{score}^{\alpha}\cdot \text{IoU}^{\beta}, \qquad \alpha=0.5,\ \beta=6.0$$

The big `β=6` makes IoU dominate: localization quality matters most, but class score breaks ties. Procedure (all
**stop‑gradient** — assignment leaks no gradient):
1. **IoU** of every predicted box vs every GT box.
2. **align** `score^0.5 · IoU^6` (log‑space).
3. **Top‑k (k=10)** anchors per GT **and** a **spatial mask** (anchor centre inside the GT box).
4. **Duplicate resolution:** a cell matched to several GTs takes the **max‑IoU** GT.
5. **Soft class targets:** `target_scores = one_hot(label) · (align_norm · pos_overlaps)` — the cls target is
   *scaled by localization quality*, so well‑placed cells learn a stronger signal.

Anchors are cell centres `(i+0.5)·stride`; the spatial mask keeps only those inside a GT box:

> **In plain words — assigning homework.** The image is a grid of thousands of tiny "cells", each ready to predict an
> object. But only a few sit right on top of each real object. *Assignment* is the teacher handing each object's
> "homework" to the handful of cells that are both **well‑placed** (their box overlaps the object) and **confident
> about the right class**. Those chosen cells become the *positives* that actually get trained on that object; the
> rest are told "there's nothing here." This avoids thousands of cells all arguing over one object.

In [ ]:
# === Anchors inside a GT box (spatial-mask candidates) =================================
def build_anchors(levels=((8,),(16,),(32,)), H=672, W=672):
    out=[]
    for (st,) in levels:
        ys=(np.arange(H//st)+0.5)*st; xs=(np.arange(W//st)+0.5)*st; gx,gy=np.meshgrid(xs,ys)
        out.append(np.stack([gx.ravel(),gy.ravel()],1))
    return [o.astype(np.float32) for o in out]
ex = SAMPLES[2]; anc32 = build_anchors(((32,),))[0]
box = ex['boxes'][int(np.argmax((ex['boxes'][:,2]-ex['boxes'][:,0])*(ex['boxes'][:,3]-ex['boxes'][:,1])))]
gx1,gy1,gx2,gy2 = box[1]*672, box[0]*672, box[3]*672, box[2]*672
inside = (anc32[:,0]>=gx1)&(anc32[:,0]<=gx2)&(anc32[:,1]>=gy1)&(anc32[:,1]<=gy2)
fig, ax = plt.subplots(figsize=(5.4,5.4)); ax.imshow(ex['image']); ax.set_xticks([]); ax.set_yticks([])
ax.scatter(anc32[~inside,0], anc32[~inside,1], s=6, c='#bbb')
ax.scatter(anc32[inside,0], anc32[inside,1], s=24, c='#ff3b30')
draw_boxes(ax, box[None,:], None, color='#0a84ff', lw=2)
ax.set_title("anchors (stride 32) — red = inside GT box\n(spatial-mask candidates)", fontsize=10)
plt.tight_layout()
_save('08_anchors'); plt.show()

![08_anchors](assets/explainer/08_anchors.png)

<sub>Only cells whose centre falls inside the GT box can be assigned to it (the spatial constraint).</sub>

In [ ]:
# === Run the REAL TaskAlignedAssigner and visualise the assignment =====================
from losses.tal_assigner import TaskAlignedAssigner
anc = np.concatenate(build_anchors(), 0); A = anc.shape[0]
exa = SAMPLES[6]; order = np.argsort(-((exa['boxes'][:,2]-exa['boxes'][:,0])*(exa['boxes'][:,3]-exa['boxes'][:,1])))[:3]
gt_b = exa['boxes'][order]; gt_xyxy = np.stack([gt_b[:,1]*672, gt_b[:,0]*672, gt_b[:,3]*672, gt_b[:,2]*672],1).astype(np.float32)
gt_lab = (exa['classes'][order] % 39).astype(np.int64); M = 3
gt_cx=(gt_xyxy[:,0]+gt_xyxy[:,2])/2; gt_cy=(gt_xyxy[:,1]+gt_xyxy[:,3])/2; gt_w=gt_xyxy[:,2]-gt_xyxy[:,0]; gt_h=gt_xyxy[:,3]-gt_xyxy[:,1]
# realistic 'roughly-trained' predictions: box sized like the nearest GT, score gaussian-decaying from its centre
d2=(anc[:,0:1]-gt_cx[None])**2+(anc[:,1:2]-gt_cy[None])**2; near=np.argmin(d2,1); hw=gt_w[near]/2; hh=gt_h[near]/2
pd_box=np.stack([anc[:,0]-hw,anc[:,1]-hh,anc[:,0]+hw,anc[:,1]+hh],1).astype(np.float32)
sigma=(gt_w[near]+gt_h[near])/4+1e-3; pd_scores=np.full((1,A,39),0.02,np.float32)
pd_scores[0,np.arange(A),gt_lab[near]]=np.maximum(0.05, np.exp(-d2[np.arange(A),near]/(2*sigma**2)).astype(np.float32))
*_, fg = TaskAlignedAssigner(topk=10, alpha=0.5, beta=6.0)(tf.constant(pd_scores), tf.constant(pd_box[None]),
    tf.constant(anc), tf.constant(gt_lab[None]), tf.constant(gt_xyxy[None]), tf.constant(np.ones((1,M),bool)))
fg = fg.numpy()[0]
def iou_xyxy(a,b):
    x1=np.maximum(a[:,None,0],b[None,:,0]);y1=np.maximum(a[:,None,1],b[None,:,1]);x2=np.minimum(a[:,None,2],b[None,:,2]);y2=np.minimum(a[:,None,3],b[None,:,3])
    inter=np.clip(x2-x1,0,None)*np.clip(y2-y1,0,None); aa=(a[:,2]-a[:,0])*(a[:,3]-a[:,1]); ab=(b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    return inter/(aa[:,None]+ab[None,:]-inter+1e-9)
match=np.argmax(iou_xyxy(pd_box,gt_xyxy),1); cols=['#ff3b30','#0a84ff','#34c759']
fig, ax = plt.subplots(figsize=(6.2,6.2)); ax.imshow(exa['image']); ax.set_xticks([]); ax.set_yticks([])
for m in range(M):
    sel = fg & (match==m); ax.scatter(anc[sel,0], anc[sel,1], s=70, c=cols[m], edgecolors='k', linewidths=0.5, label=f"GT {m}")
    draw_boxes(ax, gt_b[m][None], None, color=cols[m], lw=2)
ax.legend(loc='lower right', fontsize=8)
ax.set_title(f"REAL TaskAlignedAssigner → {int(fg.sum())} foreground anchors\n(each GT keeps its top-10 aligned, in-box cells)", fontsize=10)
plt.tight_layout()
_save('08_assigner'); plt.show()

![08_assigner](assets/explainer/08_assigner.png)

<sub>The real assigner picks ~10 cells per GT — the ones that are both well-localized and confidently classified.</sub>

### Which prediction is compared with which target — and why

| Prediction (head) | Target | On which cells | Loss | Why |
|---|---|---|---|---|
| `box` (DFL→ltrb) | `target_bboxes` (assigned GT box) | foreground | CIoU + DFL | only responsible cells learn the box; weighted by `Σ target_scores` |
| `cls` | `target_scores` (soft one‑hot) | **all** cells | BCE | background must learn "nothing"; soft target couples cls to IoU |
| `poly_angle/dist/conf` | gathered radial `[72]` of the GT | foreground (detection rows) | BCE/L2/BCE | a polygon only exists for a cell that owns an object |
| `dist` | gathered `log_distance` of the GT | foreground **and** valid (`>−10`) | L1 | distance is only supervised where a measurement exists |

The assignment is `tf.stop_gradient` — it shapes *what* is compared, never leaking gradient itself.

## 9 · The loss — how the model learns, term by term

`losses/tal_loss.py:TaskAlignedLossExtended.__call__(feats, batch)` returns a **9‑tuple**
`(total, box, dfl, cls, dist, poly, poly_angle_raw, poly_dist_raw, poly_conf_raw)` (last three for logging). The total:

$$
\mathcal{L} = \underbrace{7.5\,\mathcal{L}_{\text{CIoU}} + 1.5\,\mathcal{L}_{\text{DFL}}}_{\text{box}}
+ \underbrace{0.5\,\mathcal{L}_{\text{cls}}}_{\text{class}} + \underbrace{1.0\,\mathcal{L}_{\text{dist}}}_{\text{distance}}
+ 0.5\underbrace{\big(0.4\,\mathcal{L}_{\angle} + 0.45\,\mathcal{L}_{\text{r}} + 0.2\,\mathcal{L}_{\text{conf}}\big)}_{\text{polygon}}
$$

Each head speaks a different "language", so each gets a tailored loss. We walk through all of them.

> **In plain words — a panel of examiners.** Remember from the primer: a *loss* is one number for "how wrong". Here
> each head is graded by its **own examiner** using the measure that fits its job — the box examiner uses overlap
> (IoU/CIoU), the class examiner uses a confidence penalty (BCE), the distance examiner uses plain error (L1), the
> polygon examiners grade each of the 24 spokes. Each examiner's score is multiplied by an importance **gain** and the
> scores are **added into one total**. Training then nudges the weights to lower that total — so *all* the heads
> improve together.

In [ ]:
# === Loss-flow diagram =================================================================
fig, ax = plt.subplots(figsize=(13.5, 5.0)); ax.set_xlim(0,100); ax.set_ylim(0,44); ax.axis('off')
ax.set_title("Each head → its loss → ×gain → total", fontsize=12, weight='bold')
rows=[("box (DFL)","CIoU + DFL","7.5 / 1.5",37,"block"),("cls","BCE (soft target)","0.5",30,"out"),
      ("dist","L1 (log-scale)","1.0",23,"out"),("poly_angle","BCE (sub-bin offset)","0.4",16,"concat"),
      ("poly_dist","L2 (softplus)","0.45",9,"concat"),("poly_conf","BCE (all 24 bins)","0.2",2,"concat")]
for h,l,g,yy,k in rows:
    node(ax, 2, yy, 16, 5.6, h, "", k, fs=9); node(ax, 25, yy, 22, 5.6, l, "", "op", fs=8.5)
    ax.text(56, yy+2.8, "×"+g, fontsize=9.5, color="#9b2d5e", ha='center', weight='bold')
    arrow(ax, 18, yy+2.8, 25, yy+2.8); arrow(ax, 47, yy+2.8, 62, 22)
node(ax, 64, 17, 22, 9, "TOTAL", "(poly terms ×poly_gain 0.5)", "out", fs=10)
plt.tight_layout()
_save('09_flow'); plt.show()

![09_flow](assets/explainer/09_flow.png)

<sub>Six head-specific losses, each scaled by a gain, summed into one scalar to backprop.</sub>

### 9.1 Box — CIoU (overlap + centre + aspect)
A box has 4 numbers, but treating them as independent regressions ignores that they form a *shape*. **CIoU** scores
the boxes as wholes:

$$\mathcal{L}_{\text{CIoU}} = 1 - \Big(\text{IoU} - \tfrac{\rho^2}{c^2} - \alpha v\Big)$$

- **IoU** — area overlap (the core signal).
- **`ρ²/c²`** — squared centre distance over the enclosing‑box diagonal: pulls centres together even when boxes don't
  yet overlap (where plain IoU gives zero gradient).
- **`α·v`** — an aspect‑ratio penalty: matches width/height proportions.

In [ ]:
# === CIoU components diagram + CIoU vs IoU (REAL _ciou_loss) ============================
from losses.tal_loss import _ciou_loss
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
a=ax[0]; a.set_xlim(0,1); a.set_ylim(0,1); a.set_aspect('equal'); a.invert_yaxis(); a.axis('off'); a.set_title("CIoU geometry", fontsize=10)
gt=[0.30,0.30,0.70,0.65]; pr=[0.42,0.40,0.85,0.80]
a.add_patch(Rectangle((gt[0],gt[1]),gt[2]-gt[0],gt[3]-gt[1], fill=False, ec='#ff3b30', lw=2, label='GT'))
a.add_patch(Rectangle((pr[0],pr[1]),pr[2]-pr[0],pr[3]-pr[1], fill=False, ec='#2e8b57', lw=2, label='pred'))
ex0=[min(gt[0],pr[0]),min(gt[1],pr[1]),max(gt[2],pr[2]),max(gt[3],pr[3])]
a.add_patch(Rectangle((ex0[0],ex0[1]),ex0[2]-ex0[0],ex0[3]-ex0[1], fill=False, ec='#9b2d5e', lw=1, ls='--'))
gc=((gt[0]+gt[2])/2,(gt[1]+gt[3])/2); pc=((pr[0]+pr[2])/2,(pr[1]+pr[3])/2)
a.plot([gc[0],pc[0]],[gc[1],pc[1]],'-o',c='#0a84ff',ms=4); a.text((gc[0]+pc[0])/2,(gc[1]+pc[1])/2-0.04,"ρ (centre dist)",color='#0a84ff',fontsize=8)
a.plot([ex0[0],ex0[2]],[ex0[1],ex0[3]],':',c='#9b2d5e'); a.text(ex0[0]+0.02,ex0[3]-0.02,"c (enclosing diag)",color='#9b2d5e',fontsize=8)
a.legend(loc='upper left', fontsize=8)
ious=[]; closs=[]; gtb=tf.constant([[40.,40.,120.,120.]])
for dx in np.linspace(0,90,60):
    pb=tf.constant([[40.+dx,40.,120.+dx,120.]], tf.float32)
    inter=max(0,120-max(40,40+dx))*80; ious.append(inter/(80*80*2-inter)); closs.append(float(_ciou_loss(pb,gtb)[0]))
ax[1].plot(ious, closs, lw=2, color='#9b2d5e'); ax[1].set_xlabel("IoU"); ax[1].set_ylabel("CIoU loss")
ax[1].set_title("CIoU loss vs IoU (REAL _ciou_loss)\nhigher overlap → lower loss", fontsize=10)
plt.tight_layout()
_save('09_ciou'); plt.show()

![09_ciou](assets/explainer/09_ciou.png)

<sub>CIoU adds centre-distance and aspect terms on top of IoU, giving gradient even when boxes barely overlap.</sub>

### 9.2 Box — DFL (Distribution Focal Loss)
Instead of regressing each box side to a single number, the head predicts a **probability distribution over 16 bins**
per side; the decoded side is the distribution's mean (§7). DFL trains that distribution: the continuous target
`t` (in cell units) lands between integer bins `⌊t⌋` and `⌈t⌉`, and the loss is the **negative log‑likelihood of a
linear interpolation** between them:

$$\mathcal{L}_{\text{DFL}} = -\big[(\lceil t\rceil - t)\log p_{\lfloor t\rfloor} + (t-\lfloor t\rfloor)\log p_{\lceil t\rceil}\big]$$

Why a distribution? It lets the box edge be **sub‑pixel precise** and lets the network express *uncertainty*
(a sharp peak = confident edge; a flat spread = ambiguous edge). Targets are clipped to `[0, reg_max−1.001]`.

In [ ]:
# === DFL target = soft label on two bins ===============================================
fig, ax = plt.subplots(figsize=(7.5, 3.2)); tgt=5.3; bins=np.arange(16)
wl=1-(tgt-np.floor(tgt)); wr=tgt-np.floor(tgt); w=np.zeros(16); w[5]=wl; w[6]=wr
ax.bar(bins, w, color='#0a84ff'); ax.axvline(tgt, color='r', lw=2)
ax.set_title(f"DFL target t={tgt} → soft label on bins 5 & 6  (weights {wl:.1f} / {wr:.1f})\nloss = −Σ w·log p   pushes the predicted distribution toward t", fontsize=10)
ax.set_xlabel("DFL bin"); ax.set_ylabel("target weight")
plt.tight_layout()
_save('09_dfl_target'); plt.show()

![09_dfl_target](assets/explainer/09_dfl_target.png)

<sub>DFL turns a continuous box-edge target into a 2-bin soft label and maximises the log-likelihood there.</sub>

### 9.3 Classification — BCE on *soft* targets
A per‑class **binary cross‑entropy** (sigmoid, multi‑label style) over all `C` classes, on **all** cells (so
background learns to output near‑zero). The target is not a hard 1 — it's the **soft** `target_scores` from the
assigner (`one_hot · align_norm · IoU`). This is the heart of *task‑aligned* learning: a cell's classification target
*equals its localization quality*, so the two heads are trained to agree (confident ⇔ well‑placed). Distance‑only rows
(`ignore_bg=1`) mask this loss to foreground.

### 9.4 Distance — L1 on log‑scale
`|pred − target|` on **log‑distance**, masked to foreground cells with a valid measurement (`target > −10` sentinel).
Log‑scale makes the relative error uniform (a 0.2 m error matters more at 0.5 m than at 10 m); decode is `exp` + clip.

### 9.5 Polygon — three per‑bin losses
- **angle** — BCE on the sub‑bin offset (target `∈[0,1)`), averaged over **valid** vertices only.
- **dist** — `(target − softplus(pred))²`, averaged over **valid** vertices only.
- **conf** — BCE on per‑bin validity, averaged over **ALL 24 bins** (occupied→1, empty→0).

Why is conf over *all* bins while angle/dist are masked to valid ones? Because angle/dist are *undefined* on empty
bins (no vertex), but **conf is the decode gate** — it must learn to say "no vertex here". Training conf only on
occupied bins (the pre‑2026‑06‑11 form) meant empty bins never got a negative signal: their conf drifted above the
0.4 threshold while their distance stayed untrained → the **"spiky / star" polygon artifact**. Training over all 24
bins fixed it.

In [ ]:
# === Polygon conf: masked (buggy) vs all-bins (fixed) intuition ========================
fig, ax = plt.subplots(1, 2, figsize=(12, 4.0)); thb=np.radians(np.arange(24)*15)
real_conf=np.zeros(24); real_conf[[0,3,6,9,12,15,18,21]]=1   # a sparse object: only 8 real bins
# buggy: empty bins drift to ~0.5 (above 0.4 gate) with untrained dist -> spikes
drift=real_conf.copy().astype(float); drift[drift==0]=0.55; r_buggy=np.where(drift>=0.4, 0.4+0.3*np.random.default_rng(1).random(24), 0)
for a,(title,conf,col) in zip(ax,[("masked conf (buggy): empty bins drift >0.4 → spikes",r_buggy,'#ff3b30'),
                                   ("all-bins conf (fixed): empty bins pushed to 0",np.where(real_conf>0,0.5,0),'#34c759')]):
    pts=[]
    for i in range(24):
        rr = conf[i] if conf[i]>0 else 0
        pts.append((0.5+rr*math.cos(thb[i]), 0.5+rr*math.sin(thb[i])))
    P=np.array([p for p,c in zip(pts,conf) if c>0])
    if len(P)>2: a.add_patch(MplPolygon(P, closed=True, fill=True, fc=col, ec=col, alpha=0.35, lw=1.6))
    a.plot(0.5,0.5,'k+',ms=10); a.set_xlim(0,1); a.set_ylim(0,1); a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([]); a.set_title(title, fontsize=9.5)
plt.suptitle("Why polygon conf trains over ALL 24 bins (2026-06-11 fix)", fontsize=11, weight='bold')
plt.tight_layout()
_save('09_polyconf'); plt.show()

![09_polyconf](assets/explainer/09_polyconf.png)

<sub>Empty bins must see a negative signal or their conf drifts above the gate and emits phantom spikes.</sub>

In [ ]:
# === Run the REAL loss on a real one-image batch =======================================
from losses.tal_loss import TaskAlignedLossExtended
exl = SAMPLES[6]; M = exl['boxes'].shape[0]
radial72 = np.stack([radial_encode(exl['boxes'][i], exl['polys'][i]).reshape(-1) for i in range(M)]).astype(np.float32)
batch = {"bbox": tf.constant(exl['boxes'][None]), "classes": tf.constant((exl['classes']%39)[None].astype(np.int64)),
         "n_gt": tf.constant([M], tf.int64), "polygons": tf.constant(radial72[None]),
         "log_distance": tf.constant(np.full((1,M), -10.0, np.float32)), "ignore_bg": tf.constant([0], tf.int64)}
img = (tf.cast(tf.constant(exl['image']), tf.float32)/255.0)[None]
raw = model(img, training=False)
vals = [float(v) for v in TaskAlignedLossExtended(num_classes=39)(raw, batch)]
total,box,dfl,cls,dist,poly = vals[:6]
fig, ax = plt.subplots(figsize=(8,3))
ax.bar(['box·7.5','dfl·1.5','cls·0.5','dist·1.0','poly·0.5'], [box,dfl,cls,dist,poly],
       color=['#9b2d5e','#b6791f','#2e8b57','#1f8a8a','#0a84ff'])
for i,v in enumerate([box,dfl,cls,dist,poly]): ax.text(i,v,f"{v:.2f}",ha='center',va='bottom',fontsize=9)
ax.set_title(f"REAL loss on one image · TOTAL = {total:.3f}  (untrained weights)", fontsize=11, weight='bold')
plt.tight_layout()
_save('09_real_loss'); plt.show()

![09_real_loss](assets/explainer/09_real_loss.png)

<sub>The actual TaskAlignedLossExtended run on a real image — the weighted components that sum to the total.</sub>

### 9.6 The "task‑aligned" weighting & normalization
Box CIoU and DFL aren't averaged uniformly — each cell's box loss is **weighted by its `Σ target_scores`** (its
alignment quality) and divided by `target_scores_sum`. So a cell that is both well‑classified and well‑localized
moves the box gradient *more* than a marginal one — the box and class heads are pulled into agreement. Denominators
differ by design (and are all‑reduced across GPUs via `_replica_sum`):

| Term | Denominator | Vertex reduction |
|---|---|---|
| box CIoU / DFL | `target_scores_sum`, per‑cell weighted by `Σ target_scores` | — |
| cls | `target_scores_sum` | — |
| distance | `num_objs` | — |
| polygon angle / dist | `num_objs` | mean over **valid** vertices |
| polygon conf | `num_objs` | mean over **all 24** bins |

## 10 · Updating the weights — SGD, warmup, cosine, EMA

The optimizer is `optimizers/sgd_warmup.py:SGDTorch` — **SGD with Nesterov momentum** and a few deliberate twists.
Intuition for each piece:

- **Momentum (0.937).** Average recent gradients so updates have inertia — they roll through small bumps and noisy
  mini‑batches instead of zig‑zagging. **Nesterov** evaluates the gradient *after* the momentum step (a "look‑ahead"),
  which corrects overshoot and converges a little faster.
- **Decoupled weight decay (0.0005).** Shrink weights toward zero each step (`w ← w(1 − lr·wd)`) **separately** from
  the gradient (AdamW‑style), and **only on conv kernels** — applying it to BN scales or biases would fight
  normalization. This is regularization, not part of the loss gradient.
- **Cosine LR decay (0.01 → ~0).** Start with a large step to explore, anneal smoothly to a tiny step to settle into a
  minimum. `decay_steps = 635,400 = 2118×300`.
- **Three parameter groups + warmup** — explained next.

### 10.1 Why three parameter groups?
Variables behave differently, so they're split by name into **BN** (`gamma/beta/moving_mean/variance`), **bias**, and
**weights** (`kernel`). Weight decay applies **only to weights**. During **warmup** the groups get *opposite* LR
ramps, because at step 0 the BatchNorm running statistics are garbage and a big weight update would diverge:

> **In plain words.** *Momentum* is a heavy ball: it keeps rolling in a consistent direction and ignores small bumps,
> so training is smoother and faster. The *learning rate* is how big a step we take — too big and we overshoot the
> valley, too small and we crawl. *Warmup* means starting with tiny steps so we don't lurch while the network is still
> raw, then speeding up; *cosine decay* means slowing back down near the end to settle precisely into the minimum.

In [ ]:
# === Three param groups + warmup behaviour =============================================
fig, ax = plt.subplots(figsize=(13, 4.4)); ax.set_xlim(0,100); ax.set_ylim(0,34); ax.axis('off')
ax.set_title("SGDTorch · 3 parameter groups (split by variable name)", fontsize=12, weight='bold')
groups=[("WEIGHTS (kernel)","weight decay ✓\nwarmup LR: 0 → base (ramps UP)","block",24),
        ("BIAS","weight decay ✗\nwarmup LR: 0.1 → base (ramps DOWN)","conv",14),
        ("BN (γ,β,μ,σ²)","weight decay ✗\nwarmup LR: 0.1 → base (ramps DOWN)","pool",4)]
for t,sub,k,yy in groups:
    node(ax, 2, yy, 30, 8.5, t, sub, k, fs=9.5)
ax.text(40, 17, "Why opposite ramps?\n\n• early on, BN running stats are unreliable → keep weight\n  updates tiny (ramp UP from 0) so features stabilise first\n• biases/BN start at a higher LR (0.1, ~10× base) to adapt\n  fast, then settle down to the schedule\n• momentum also warms 0.8 → 0.937 (trust history less early)",
        fontsize=9.5, va='center', bbox=dict(boxstyle="round", fc="#fffbe8", ec="#b6791f"))
plt.tight_layout()
_save('10_groups'); plt.show()

![10_groups](assets/explainer/10_groups.png)

<sub>Weights ramp up from 0 (don't over-update before BN settles); bias/BN ramp down from a high start.</sub>

In [ ]:
# === LR schedule, momentum, per-group warmup LR (REAL CosineDecay + SGDTorch._effective_lr) ===
from optimizers.sgd_warmup import SGDTorch
warmup=6354; cosine=tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=0.01, decay_steps=635400, alpha=0.01)
opt=SGDTorch(lr_fn=lambda s: cosine(s), momentum=0.937, momentum_start=0.8, warmup_steps=warmup, bias_lr_scale=0.1)
steps=np.arange(0,635400,2000); base=np.array([float(cosine(int(s))) for s in steps]); warm=np.clip(steps/warmup,0,1)
wsteps=np.arange(0,warmup,50); tt=np.clip(wsteps/warmup,0,1)
lr_w=[float(opt._effective_lr(tf.constant(0.01),tf.constant(float(t)),2)) for t in tt]
lr_bn=[float(opt._effective_lr(tf.constant(0.01),tf.constant(float(t)),0)) for t in tt]
fig, ax = plt.subplots(1, 3, figsize=(16, 3.4))
ax[0].plot(steps, base*warm, color='#0a84ff', lw=2); ax[0].axvline(warmup, color='#999', ls='--')
ax[0].set_title("LR: linear warmup → cosine decay (full 635k steps)"); ax[0].set_xlabel("step")
ax[1].plot(steps, 0.8+warm*(0.937-0.8), color='#9b2d5e', lw=2); ax[1].axvline(warmup, color='#999', ls='--')
ax[1].set_title("momentum warmup 0.8 → 0.937"); ax[1].set_xlabel("step")
ax[2].plot(wsteps, lr_w, label='weights (↑)', color='#2e8b57', lw=2); ax[2].plot(wsteps, lr_bn, label='BN/bias (↓)', color='#ff9500', lw=2)
ax[2].set_title("per-group LR during warmup (REAL _effective_lr)"); ax[2].set_xlabel("step"); ax[2].legend(fontsize=8)
plt.tight_layout()
_save('10_schedules'); plt.show()

![10_schedules](assets/explainer/10_schedules.png)

<sub>Real schedules: overall warmup→cosine LR, momentum warmup, and the opposite per-group warmup ramps.</sub>

### 10.2 The per‑step update (the real code path)
For each gradient/variable, `SGDTorch.apply_gradients` does (`optimizers/sgd_warmup.py`):
```
group  = classify(var.name)                 # 0=BN, 1=bias, 2=weights
eff_lr = effective_lr(base_lr, t, group)    # warmup ramp per group
if group == 2:  var *= (1 - eff_lr*wd)       # decoupled weight decay (weights only)
v       = μ·v + g                            # momentum accumulator
update  = μ·v + g  if nesterov else v        # Nesterov look-ahead
var    -= eff_lr * update                    # the step
```
Under multi‑GPU, gradients are **SUM all‑reduced** across replicas (`_all_reduce_gradients`) and the loss normalizers
are global, so a multi‑GPU run is numerically identical to single‑GPU. Gradients are global‑norm clipped after the
all‑reduce.

### 10.3 EMA — a smoothed copy for evaluation
`optimizers/ema.py` keeps a **shadow** copy of the weights, updated every step as `shadow ← decay·shadow +
(1−decay)·w`. The averaged weights sit in a flatter, better‑generalizing spot than the noisy live weights, so they're
used **for evaluation** (swapped in, then swapped back — crash‑safe). The **decay is dynamic**:
`min(0.9999, (1+step)/(10+step))` — near 0 early (trust the rapidly‑improving live weights) rising to 0.9999 (average
over a long horizon once training matures).

In [ ]:
# === EMA decay curve + swap mechanics ==================================================
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
steps=np.arange(1,30000); ax[0].plot(steps, np.minimum(0.9999,(1+steps)/(10+steps)), lw=2, color='#1f8a8a')
ax[0].axhline(0.9999, color='#999', ls='--'); ax[0].set_title("EMA dynamic decay  min(0.9999,(1+t)/(10+t))", fontsize=10)
ax[0].set_xlabel("step"); ax[0].set_ylabel("decay")
a=ax[1]; a.set_xlim(0,100); a.set_ylim(0,30); a.axis('off'); a.set_title("EMA swap around evaluation", fontsize=10)
node(a, 2, 18, 26, 8, "LIVE weights", "(trained by SGD)", "block", fs=9)
node(a, 2, 4, 26, 8, "SHADOW (EMA)", "shadow←d·shadow+(1−d)·w", "out", fs=8.5)
node(a, 60, 11, 34, 9, "evaluation", "uses EMA weights\nswap_in → eval → swap_out", "op", fs=9)
arrow(a, 28, 8, 60, 13, "swap_in", lcol="#2e8b57"); arrow(a, 60, 17, 28, 22, "swap_out (restore)", lcol="#9b2d5e")
plt.tight_layout()
_save('10_ema'); plt.show()

![10_ema](assets/explainer/10_ema.png)

<sub>Dynamic decay trusts live weights early, averages heavily later. EMA weights are swapped in only for eval.</sub>

## 11 · Runtime & performance — the problems each trick solves

Applied in `scripts/run_train.py:_apply_runtime_config` (+ pipeline tricks). For each: the **problem**, the **fix**,
the **trade‑off**.

| Optimization | Problem it solves | Fix | Trade‑off |
|---|---|---|---|
| **`bfloat16`** | fp32 is memory/bandwidth heavy | compute bf16; heads + loss float32 | bf16 has fp32's exponent range → **no loss scaling**; slightly less mantissa |
| **(fp16 rejected)** | fp16 underflows without loss scaling | raise `NotImplementedError` | must use bf16 or fp32 |
| **XLA JIT** | many tiny kernel launches | `set_jit(True)` fuses ops | compile time; re‑compiles on new shapes |
| **Graph mode** | Python/eager overhead per op | `@tf.function` on train/val step | harder to debug (use `--debug` for eager) |
| **Thread caps** | TF sees all cores but is cgroup‑capped → oversubscription/thrash | cap `inter/intra/private_threadpool` | must match the real core quota |
| **Strategy** | MirroredStrategy var‑wrapping overhead on 1 GPU | `one_device` for 1 GPU; Mirrored for N | identical numerics either way |
| **SkipDecoding** | decoded images are MBs in the shuffle buffer | keep JPEG bytes (KB), decode in map | decode cost moves into the parallel map |
| **`deterministic=False` + prefetch** | head‑of‑line blocking; GPU starves | unordered maps + `prefetch(AUTOTUNE)` | sample order not seed‑reproducible |
| **GPU colour aug** | HSV/Albu burned CPU tf.data threads | move into `train_step` | uses GPU time (cheap there) |

### 11.1 Why bf16 and not fp16
`float16` has a tiny exponent range, so small gradients **underflow to zero** unless you scale the loss up and back
down (loss scaling). This codebase's custom `SGDTorch` + bare `GradientTape` has **no loss scaling**, so `float16` is
explicitly **rejected**. `bfloat16` keeps `float32`'s 8‑bit exponent (same dynamic range) and just drops mantissa
bits — gradients don't underflow, **no loss scaling needed**. The 6 prediction convs and the loss stay `float32` for
numerically‑sensitive softmax/BCE.

> **In plain words — a kitchen.** The GPU is a fast chef; the CPU pipeline is the prep station chopping vegetables.
> If the chef waits for each batch of veg, they're idle half the time. *Prefetching* means the prep station always has
> the next batch ready, so the chef never stops. *bfloat16* is like writing numbers with fewer decimal places to work
> faster — fine as long as we don't lose the important digits (which is why the delicate final layers stay full
> precision). *XLA* is writing out the whole recipe once and following it fast, instead of re‑reading each step.

In [ ]:
# === train_step timeline: data-wait vs compute, prefetch overlap =======================
fig, ax = plt.subplots(figsize=(13, 3.6)); ax.set_xlim(0,100); ax.set_ylim(0,22); ax.axis('off')
ax.set_title("Why prefetch + GPU colour aug: overlap data prep with compute", fontsize=12, weight='bold')
ax.text(2,18.5,"WITHOUT prefetch:", fontsize=9.5, color='#9b2d5e', weight='bold')
for x,w,c,l in [(16,14,'#ffd1c4','CPU data'),(30,10,'#cfe0f5','GPU step'),(40,14,'#ffd1c4','CPU data'),(54,10,'#cfe0f5','GPU step')]:
    ax.add_patch(Rectangle((x,15),w,3, fc=c, ec='#777')); ax.text(x+w/2,16.5,l,ha='center',fontsize=7)
ax.text(70,16.5,"GPU idle during data ✗", fontsize=8, color='#9b2d5e')
ax.text(2,10.5,"WITH prefetch(AUTOTUNE):", fontsize=9.5, color='#2e8b57', weight='bold')
for x,w,c,l in [(16,14,'#ffd1c4','CPU data'),(30,40,'#cfe0f5','GPU steps (continuous)')]:
    ax.add_patch(Rectangle((x,6),w,3, fc=c, ec='#777')); ax.text(x+w/2,7.5,l,ha='center',fontsize=7)
for x in [30,42,54,66]: ax.add_patch(Rectangle((x,2.5),11,2.6, fc='#ffe9c4', ec='#b6791f'))
ax.text(48,3.7,"CPU prepares next batch while GPU computes (overlap)", ha='center', fontsize=7.5, color='#b6791f')
ax.text(72,7.5,"data_wait_ms → ~0 ✓", fontsize=8, color='#2e8b57')
plt.tight_layout()
_save('11_timeline'); plt.show()

![11_timeline](assets/explainer/11_timeline.png)

<sub>prefetch overlaps the next batch's CPU prep with the current GPU step; `train/data_wait_ms` measures the gap.</sub>

### 11.2 The three measured CPU‑bottleneck fixes
On the 13‑core‑capped cloud host, three pipeline changes shifted the dominant costs off the capped tf.data
threadpool (measured per image):
- **pre‑resize to 672² before copy‑paste** — composite at 672² instead of full‑res (~**18 ms·core/img**).
- **mosaic canvas formulation** — one warp per mosaic instead of an intermediate 2× canvas resize (~**54 ms·core/img**).
- **colour aug → GPU** — HSV/Albumentations out of the parser (~**20 ms·core/img**).

`train/data_wait_ms` vs `train/step_time_ms` (in `trainer.py`) tells you whether you're CPU‑ (data) or GPU‑ (compute)
bound. **Epoch semantics:** the stream is infinite; the trainer runs exactly `steps_per_loop = 2118` steps/epoch from
one persistent iterator, so epoch boundaries, the cosine schedule, and checkpoint intervals all line up by construction.

In [ ]:
# === CPU/GPU/precision runtime map =====================================================
fig, ax = plt.subplots(figsize=(13, 4.2)); ax.set_xlim(0,100); ax.set_ylim(0,30); ax.axis('off')
ax.set_title("Where the work runs, and at what precision", fontsize=12, weight='bold')
node(ax, 1, 18, 30, 9, "CPU · tf.data (capped threads)", "SkipDecoding · uint8 geometry\ndeterministic=False · prefetch", "block", fs=8.5)
node(ax, 35, 18, 30, 9, "GPU · mixed bfloat16", "backbone/decoder/head bf16\n6 pred convs + loss → float32", "out", fs=8.5)
node(ax, 69, 18, 30, 9, "graph + XLA", "@tf.function steps\nset_jit fusion · one_device", "conv", fs=8.5)
arrow(ax, 31, 22.5, 35, 22.5); arrow(ax, 65, 22.5, 69, 22.5)
node(ax, 20, 4, 60, 8, "no loss scaling (bf16) · grads all-reduced + clipped · EMA swapped for eval", "", "op", fs=9)
arrow(ax, 50, 18, 50, 12)
plt.tight_layout()
_save('11_runtime'); plt.show()

![11_runtime](assets/explainer/11_runtime.png)

<sub>CPU does cheap uint8 geometry; GPU runs bf16 compute with float32 heads/loss; steps are graph-compiled.</sub>

## 12 · Recap, glossary & where everything lives

**The whole story in one breath.** Real photos with box+polygon GT flow through a CPU `tf.data` pipeline (copy‑paste →
mosaic → perspective → flip), where each polygon becomes a fixed **`[24,3]` radial target**. A 672² `uint8` batch
(detection 128 + distance 16) reaches the GPU, gets colour‑augmented and normalised, and passes through
**CSPDarkNetV8‑S → FPN‑PAN → 6 heads** producing 9,261 per‑cell predictions. **Task‑aligned assignment** turns sparse
GT into dense soft targets; a **multi‑term loss** (CIoU+DFL box, soft‑target BCE cls, log‑L1 distance, 3‑part polygon)
scores them; **SGD+Nesterov** with warmup→cosine updates the weights and an **EMA** shadow is kept for evaluation —
all under bf16 + XLA + graph mode.

**Glossary.** **DFL** = Distribution Focal Loss (box side as a 16‑bin distribution). **TAL** = Task‑Aligned Learning
(assignment by `score^0.5·IoU^6`). **PolyYOLO radial** = polygon as 24 `(dist, angle, conf)` bins. **C2f** =
Cross‑Stage‑Partial block. **SPPF** = Spatial Pyramid Pooling‑Fast. **CIoU** = Complete‑IoU (overlap + centre +
aspect). **EMA** = Exponential Moving Average of weights. **ignore_bg** = flag masking class loss for distance rows.

| Concept | File |
|---|---|
| Pipeline assembly, two‑stream merge | `data_pipeline/input_reader.py` |
| Copy‑Paste / Mosaic / perspective / flip / resample | `data_pipeline/{copy_paste,mosaic,augmentations}.py` |
| Polygon → radial target | `data_pipeline/yolo_parser.py:_preprocess_polygons_v2` |
| GPU colour aug | `data_pipeline/batch_color_aug.py` |
| Backbone / decoder / head / assembly / NMS | `models/{backbone,decoder,head,yolo_v8,detection_generator}.py` |
| Assignment & losses | `losses/{tal_assigner,tal_loss,polygon_loss,distance_loss}.py` |
| Optimizer & EMA | `optimizers/{sgd_warmup,ema}.py` |
| Training loop & runtime | `train/{task,trainer}.py`, `scripts/run_train.py` |
| Config (dataclasses + YAML) | `configs/{model_config,yaml_loader}.py`, `experiments/yolo/*.yaml` |

> **Design register.** Before "fixing" anything surprising (additive HSV, polygon conf‑over‑all‑bins, the `‑1.0`
> sentinel, mosaic canvas formulation, warmup ramp direction), read `docs/design_register.md` — these are deliberate,
> tested decisions, several visualised above.

*Every figure was rendered by the project's real code on real images (saved under `notebooks/assets/explainer/`).
Model weights here are randomly initialised, so predicted **values** are placeholders — every **shape, target, loss
and schedule** is exactly what training uses. Re‑run any cell to regenerate its figure.*